### Single-particle tracking pipeline for 2D data
Input: 
> Set of localizations, currently accepting ThunderStorm format \
> You need to change some of the parameters here and there; the text before the code cells will tell you \

Output:  
 >   1. Linked trajectories
  >  2. MSD analysis/graphs with trackpy
   > 3. MSD fits (D*, alpha) with graphs
>

Dependencies:
> matplotlib, pandas, numpy \
> trackpy, scipy, seaborn

#### Load all packages

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import trackpy as tp
from scipy.optimize import curve_fit
import seaborn as sns
import os
import warnings
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')

# --- Plot styling ---
plt.rcParams['text.usetex'] = False
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
plt.rc('legend', fontsize=14, title_fontsize=14)
plt.style.use('seaborn-v0_8-white')
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['xtick.major.size'] = 5
plt.rcParams["font.family"] = 'Arial'

#### Initialize all the functions

In [ ]:
#### Initialize all the functions
def load_locs(dir, fn, conv_factor):
    locs = pd.read_csv(dir + fn)
    locs = locs[["frame", "x [nm]", "y [nm]"]].rename(columns={"x [nm]": "x", "y [nm]": "y"})
    locs = locs.mul({'frame':1, 'x':conv_factor, 'y':conv_factor})
    locs['file'] = fn
    return locs

def track_and_filter(locs, search_range, memory, exact_len, fn):
    trajs = tp.link(locs, search_range, memory=memory, link_strategy='auto')

    new_tracks = []
    new_pid = 0

    for pid, g in trajs.groupby('particle'):
        if len(g) >= exact_len:
            g = g.sort_values('frame').iloc[:exact_len].copy()
            g['frame'] = np.arange(len(g))  # reset frames to 0,1,...,59
            g['particle'] = new_pid         # assign new unique particle ID
            new_tracks.append(g)
            new_pid += 1



    if not new_tracks:
        return trajs.iloc[0:0]  # empty DataFrame with same columns

    tracks = pd.concat(new_tracks, ignore_index=True)

    if len(tracks) > 0:
        fig, ax = plt.subplots(figsize=(7,7))
        tp.plot_traj(tracks, mpp=1, ax=ax)
        plt.axis('equal')
        ax.set_xlabel(r'x [$\mu$m]', fontsize=14)
        ax.set_ylabel(r'y [$\mu$m]', fontsize=14)
        ax.tick_params(axis='both', labelsize=12)
        ax.grid(False)
        plt.savefig(fn + '_tracks.svg')
        plt.close()

    return tracks


def calc_imsds(tracks, mpp, fps, maxlag, fn):
    im = tp.imsd(tracks, mpp, fps, max_lagtime=maxlag)
    imsr = im.round(7)
    im = imsr.T.drop_duplicates().T
    fig, ax = plt.subplots(figsize=(7,7))
    ax.plot(im.index, im, 'k-', alpha=0.3)
    ax.set(ylabel=r'MSD [$\mu$m$^2$]', xlabel='lag time $t (s)$')
    ax.set_yscale('linear')
    ax.set_xscale('linear')
    ax.set_title('iMSDs', fontsize=16, fontweight='bold')
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(False)
    plt.savefig(fn)
    plt.close()
    return im

# --- MSD models ---
def msd_2D_log(lag, D_eff, alpha, sigma):
    return np.log((4 * D_eff * np.power(lag, alpha)) + 4 * np.power(sigma, 2))


def msd_2D(lag, D_eff, alpha, sigma):
    return (4 * D_eff * np.power(lag, alpha)) + 4 * np.power(sigma, 2)


def backlund_msd_2D(lag, D_eff, alpha, sigma):
    """
    Backlund MSD model for motion blur.
    Uses elapsed lag time between frames (30 s)
    and per-slice exposure time (0.05 s) for integration.
    """
    t_exp = 0.05  # single-slice exposure time in seconds
    n = lag / t_exp  # number of exposure-length intervals in the lag
    term1 = (4 * D_eff * (t_exp ** alpha)) * (
        (np.power(n + 1, alpha + 2) + np.power(n - 1, alpha + 2) - 2 * np.power(n, alpha + 2))
        / ((alpha + 2) * (alpha + 1))
    )
    term2 = (8 * D_eff * (t_exp ** alpha)) / ((alpha + 2) * (alpha + 1))
    return term1 - term2 + 4 * (sigma ** 2)


def backlund_msd_2D_log(lag, D_eff, alpha, sigma):
    return np.log(backlund_msd_2D(lag, D_eff, alpha, sigma))


# --- Fit iMSDs ---
def fit_imsds(im, t_exp, fn, fitback):
    """
    Fits iMSD curves using either standard MSD or Backlund MSD model.
    `t_exp` is the per-slice exposure (0.05 s).
    """
    fits = pd.DataFrame(index=im.index)
    xdata = im.index.to_numpy()  # lag times (30, 60, 90 s, etc.)
    bounds = ([1e-6, 0.01, 0.001], [0.1, 2.0, 0.1])
    p0 = [0.001, 0.5, 0.02]
    all_params = []

    for i in im.columns:
        ydata = im[i].to_numpy()
        try:
            if fitback:
                popt, _ = curve_fit(
                    backlund_msd_2D_log, xdata, np.log(ydata),
                    bounds=bounds, p0=p0, maxfev=5000
                )
                fits["Fit of " + str(i)] = backlund_msd_2D(xdata, *popt)
            else:
                popt, _ = curve_fit(
                    msd_2D_log, xdata, np.log(ydata),
                    bounds=bounds, p0=p0, maxfev=5000
                )
                fits["Fit of " + str(i)] = msd_2D(xdata, *popt)

            all_params.append({"D_eff": popt[0], "alpha": popt[1], "sigma": popt[2]})
        except Exception:
            continue

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(7, 7))
    im.plot(style="k-", alpha=0.2, linewidth=2, ax=ax)
    fits.plot(ax=ax, style="k--", alpha=0.6, linewidth=2)
    ax.set_yscale("linear")
    ax.set_xscale("linear")
    ax.set_ylabel("MSD (µm²)", fontsize=14)
    ax.set_title("Fit to 2D MSD", fontsize=16, fontweight="bold")
    ax.tick_params(axis="both", labelsize=12)
    ax.grid(False)
    lined = Line2D([0], [0], label="Data", color="k", linestyle="-")
    linef = Line2D([0], [0], label="Fit", color="k", linestyle="--")
    ax.legend(handles=[lined, linef])
    plt.savefig(fn)
    plt.close()

    return pd.DataFrame(all_params)

# --- Violin plots ---
def plot_violins(params, fn):
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib import ticker as mticker

    params = params.copy()
    
    # Filter out unphysical or failed fits
    params = params[(params['D_eff'] > 1e-5) & 
                    (params['alpha'] > 0.01) & 
                    (params['sigma'] > 0.008)]

    
    params['log_D_eff'] = np.log10(params['D_eff'])

    fig, axes = plt.subplots(1, 3, figsize=(15,5))
    colors = sns.color_palette("Purples", 3)

    # --- D_eff ---
    #sns.violinplot(y=params['log_D_eff'], ax=axes[0], color=colors[1], inner='box')
    #sns.stripplot(y=params['log_D_eff'], ax=axes[0], color='k', size=3, alpha=0.25)
    
    # Dynamic y-axis based on filtered data
    #logD_min = np.floor(params['log_D_eff'].min())
    #logD_max = np.ceil(params['log_D_eff'].max())
    #padding = 0.2 * (logD_max - logD_min)  # 20% padding
    #axes[0].set_ylim(logD_min - padding, logD_max + padding)
    #axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))
    
    #axes[0].set_ylabel(r"$D_{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    #axes[0].set_title("Diffusion Coefficient", fontsize=16, fontweight='bold')
    #median_D = np.median(params['D_eff'])
    #axes[0].text(0.95, 0.95, f"Median: {median_D:.2e}", transform=axes[0].transAxes,
    #             fontsize=12, va='top', ha='right', fontweight='bold')
    
    #axes[0].tick_params(axis='both', labelsize=12)
    #axes[0].grid(False)


    ####TRYING TO FIX LOG AXIS SCALING ON VIOLIN PLOTS, CHANGE BACK TO ABOVE VERSION IF IT DOESNT WORK

    # --- D_eff ---
    # Use the raw D_eff, not the log10 version
    sns.violinplot(y=params['D_eff'], ax=axes[0], color=colors[1], inner='box')
    sns.stripplot(y=params['D_eff'], ax=axes[0], color='k', size=3, alpha=0.25)
    
    # Force the Y-axis to be logarithmic
    axes[0].set_yscale('log')
    
    ymin = 10**np.floor(np.log10(params['D_eff'].min()))
    ymax = 10**np.ceil(np.log10(params['D_eff'].max()))
    axes[0].set_ylim(ymin, ymax)
    
    # Scientific notation formatter for log scale
    axes[0].yaxis.set_major_formatter(mticker.LogFormatterSciNotation())
    
    axes[0].set_ylabel(r"$D_{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    axes[0].set_title("Diffusion Coefficient", fontsize=16, fontweight='bold')
    
    median_D = np.median(params['D_eff'])
    axes[0].text(0.95, 0.95, f"Median: {median_D:.2e}", transform=axes[0].transAxes,
                 fontsize=12, va='top', ha='right', fontweight='bold')

    # --- Alpha ---
    sns.violinplot(y=params['alpha'], ax=axes[1], color=colors[2], inner='box')
    sns.stripplot(y=params['alpha'], ax=axes[1], color='k', size=3, alpha=0.25)
    axes[1].set_ylabel(r"$\alpha$", fontsize=14)
    axes[1].set_title("Anomalous Exponent", fontsize=16, fontweight='bold')
    median_a = np.median(params['alpha'])
    axes[1].text(0.95, 0.95, f"Median: {median_a:.3f}", transform=axes[1].transAxes,
                 fontsize=12, va='top', ha='right', fontweight='bold')
    axes[1].tick_params(axis='both', labelsize=12)
    axes[1].grid(False)

    # --- Sigma ---
    sns.violinplot(y=params['sigma'], ax=axes[2], color=colors[1], inner='box')
    sns.stripplot(y=params['sigma'], ax=axes[2], color='k', size=3, alpha=0.25)
    axes[2].set_ylabel(r"$\sigma$ ($\mu m$)", fontsize=14)
    axes[2].set_title("Localization Error", fontsize=16, fontweight='bold')
    median_s = np.median(params['sigma'])
    axes[2].text(0.95, 0.95, f"Median: {median_s:.4f}", transform=axes[2].transAxes,
                 fontsize=12, va='top', ha='right', fontweight='bold')
    axes[2].tick_params(axis='both', labelsize=12)
    axes[2].grid(False)

    plt.tight_layout()
    fig.savefig(fn, bbox_inches='tight')
    plt.close(fig)

# --- Scan area plots
def calculate_scan_areas_and_plot(tracks, cell_id, fn_prefix):
    import numpy as np
    from scipy.spatial import ConvexHull
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    padding_fraction = 0.1
    track_ids = tracks['particle'].unique()
    scan_areas = []
    cmap = plt.cm.viridis
    all_areas = []
    fig, ax = plt.subplots(figsize=(7,7))
    
    # Compute per-track scan areas
    for pid in track_ids:
        single_track = tracks[tracks['particle']==pid]
        x, y = single_track['x'].values, single_track['y'].values
        if len(x) > 2:
            hull = ConvexHull(np.column_stack((x, y)))
            area = hull.volume # (2D hull area)
        else:
            area = 0.0
        scan_areas.append(area)
        all_areas.append(area)
    
    all_areas = np.array(all_areas)
    if len(all_areas) == 0 or not np.any(np.isfinite(all_areas)):
        return tracks  # fallback: return original tracks

    mean_area = np.mean(all_areas[np.isfinite(all_areas)])
    norm = Normalize(vmin=np.percentile(all_areas, 5), vmax=np.percentile(all_areas, 95))
    
    # Plot each track colored by area
    ax.clear()
    for pid, area in zip(track_ids, all_areas):
        single_track = tracks[tracks['particle']==pid]
        x, y = single_track['x'].values, single_track['y'].values
        ax.plot(x, y, linewidth=1.5, color=cmap(norm(area)), alpha=0.9)
    
    # Square figure limits
    xmin, xmax = np.min(tracks['x']), np.max(tracks['x'])
    ymin, ymax = np.min(tracks['y']), np.max(tracks['y'])
    x_center, y_center = (xmin + xmax)/2, (ymin + ymax)/2
    half_range = max(xmax - xmin, ymax - ymin)/2 * (1 + padding_fraction)
    ax.set_xlim(x_center - half_range, x_center + half_range)
    ax.set_ylim(y_center - half_range, y_center + half_range)

    ax.set_title(f"Cell {cell_id} Telomere Scan Areas", fontsize=16, fontweight='bold')
    ax.set_xlabel("x [μm]", fontsize=14)
    ax.set_ylabel("y [μm]", fontsize=14)
    ax.tick_params(axis='both', labelsize=12)
    ax.set_aspect('equal')
    ax.grid(False)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Scan Area (μm²)', fontsize=14)

    # Mean area text
    ax.text(0.98, 0.98, f"Mean area: {mean_area:.3f} μm²",
            transform=ax.transAxes, ha='right', va='top',
            fontsize=12, fontweight='bold', bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

    plt.tight_layout()
    plt.savefig(f"{fn_prefix}_scan_area.png")
    plt.close()

    # Map scan areas to tracks
    tracks_with_scan = tracks.copy()
    tracks_with_scan['scan_area_um2'] = tracks_with_scan['particle'].map(
        {pid: area for pid, area in zip(track_ids, scan_areas)}
    )
    tracks_with_scan['cell_id'] = cell_id

    return tracks_with_scan


# --- Scan area plot function with CSV output ---
def plot_all_cells_scan_areas_with_csv(all_tracks, all_cell_ids, fn_prefix):
    """
    Plot scan areas for all cells and save a CSV with per-track scan areas.
    
    Parameters
    ----------
    all_tracks : pd.DataFrame
        Concatenated tracks from all cells, must include 'cell_id', 'particle', 'x', 'y'.
    all_cell_ids : list of str
        List of all cell IDs included in all_tracks.
    fn_prefix : str
        File path prefix for saved plot and CSV.
    """
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize
    from scipy.spatial import ConvexHull
    import numpy as np
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    import matplotlib.cm as cm
    import pandas as pd

    padding_fraction = 0.1
    cmap = cm.get_cmap('viridis')
    scan_area_records = []  # list to store CSV rows
    area_lookup = {}

    # Compute per-track scan areas per cell
    for cell_id in all_cell_ids:
        cell_tracks = all_tracks[all_tracks['cell_id'] == cell_id]
        for pid in cell_tracks['particle'].unique():
            single_track = cell_tracks[cell_tracks['particle'] == pid]
            x, y = single_track['x'].values, single_track['y'].values
            if len(x) > 2:
                try:
                    hull = ConvexHull(np.column_stack((x, y)))
                    area = hull.volume # (2D hull area)
                except Exception:
                    area = 0.0
            else:
                area = 0.0
            area_lookup[(cell_id, pid)] = area
            scan_area_records.append({
                'cell_id': cell_id,
                'particle': pid,
                'scan_area_um2': area
            })

    # Save CSV
    scan_area_df = pd.DataFrame(scan_area_records)
    csv_filename = f"{fn_prefix}_all_cells_scan_areas.csv"
    scan_area_df.to_csv(csv_filename, index=False)
    print(f"Saved scan areas CSV: {csv_filename}")

    # Normalize for color mapping
    all_scan_areas = scan_area_df['scan_area_um2'].values
    all_scan_areas = all_scan_areas[np.isfinite(all_scan_areas)]
    mean_area = np.mean(all_scan_areas)
    norm = Normalize(vmin=np.percentile(all_scan_areas, 5), vmax=np.percentile(all_scan_areas, 95))

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8))
    for cell_id in all_cell_ids:
        cell_tracks = all_tracks[all_tracks['cell_id'] == cell_id]
        for pid in cell_tracks['particle'].unique():
            single_track = cell_tracks[cell_tracks['particle'] == pid]
            x, y = single_track['x'].values, single_track['y'].values
            area = area_lookup.get((cell_id, pid), 0.0)
            color = cmap(norm(area))
            ax.plot(x, y, linewidth=1.2, color=color, alpha=0.9)

    # Figure limits
    xmin, xmax = np.min(all_tracks['x']), np.max(all_tracks['x'])
    ymin, ymax = np.min(all_tracks['y']), np.max(all_tracks['y'])
    x_center, y_center = (xmin + xmax) / 2, (ymin + ymax) / 2
    half_range = max(xmax - xmin, ymax - ymin) / 2 * (1 + padding_fraction)
    ax.set_xlim(x_center - half_range, x_center + half_range)
    ax.set_ylim(y_center - half_range, y_center + half_range)

    ax.set_title("Telomere Scan Areas Across All Cells", fontsize=16, fontweight='bold')
    ax.text(0.98, 0.98, f"Mean scan area:\n{mean_area:.3f} μm²",
            transform=ax.transAxes, ha='right', va='top', fontsize=12,
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

    ax.set_xlabel("x [μm]")
    ax.set_ylabel("y [μm]")
    ax.set_aspect('equal')
    ax.grid(False)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Scan Area (μm²)', fontsize=14)

    plt.tight_layout()
    fig_filename = f"{fn_prefix}_all_cells_scan_areas.png"
    plt.savefig(fig_filename, dpi=300)
    plt.close()
    print(f"Saved scan area plot: {fig_filename}")


def compute_jump_distances_and_angles(tracks):
    """
    Compute frame-to-frame jump distances and consecutive-step jump angles.

    Jump angle is defined as the angle between consecutive displacement vectors:
        angle_n = angle(Δr_n, Δr_{n+1})

    Parameters
    ----------
    tracks : pd.DataFrame
        Must contain columns ['particle', 'frame', 'x', 'y'].
        x, y should be in microns.

    Returns
    -------
    jumps : pd.DataFrame
        One row per jump with:
        ['particle', 'frame', 'dx', 'dy', 'jump_um', 'angle_rad', 'angle_deg']
    """
    jump_records = []

    for pid, g in tracks.groupby('particle'):
        g = g.sort_values('frame')

        x = g['x'].values
        y = g['y'].values
        frames = g['frame'].values

        dx = np.diff(x)
        dy = np.diff(y)

        jump_um = np.sqrt(dx**2 + dy**2)

        # --- compute consecutive-step angles ---
        for i in range(len(dx) - 1):
            v1 = np.array([dx[i], dy[i]])
            v2 = np.array([dx[i+1], dy[i+1]])

            norm1 = np.linalg.norm(v1)
            norm2 = np.linalg.norm(v2)

            if norm1 == 0 or norm2 == 0:
                angle_rad = np.nan
            else:
                cos_theta = np.dot(v1, v2) / (norm1 * norm2)
                cos_theta = np.clip(cos_theta, -1.0, 1.0)
                angle_rad = np.arccos(cos_theta)

            jump_records.append({
                'particle': pid,
                'frame': frames[i+1],  # angle centered on second step
                'dx': dx[i+1],
                'dy': dy[i+1],
                'jump_um': jump_um[i+1],
                'angle_rad': angle_rad,
                'angle_deg': np.degrees(angle_rad)
            })

    jumps = pd.DataFrame(jump_records)
    return jumps


### Load files
> change basedir to folder for this project \
> change dirn to folder with the localization files to be analyzed \
> change outname to folder to be used to save the outputs

In [ ]:
basedir = ""  # change to directory for project
dirn = basedir + "" # folder with localizations
filenames = os.listdir(dirn)
outname = basedir+''  # create this folder in your directory for outputs
print(filenames)

### Load localizations and track things
>Change "conv_factor" to your pixel size, assuming input data is in pixels (if in um, set to 1) \
>"fitback" : fit using Backlund equation (1) or not (0)
> "max_dist" : maximum allowed step size of a particle (in um) \
> "max_off" : maximum allowed off frames to stille be counted as the same particle (in frames) \
> "min_track" : minimum track length (in frames)

In [ ]:
# User parameters
condition = '' # Add condition type, ie HGPS, Control, Lonafarnib-treated, etc.
conv_factor = 0.001
fitback = 0
max_dist = 0.131*6
max_off = 3
exact_track_len = 60
fps = 0.03
max_lag = 15

all_params = pd.DataFrame()
all_imsds = pd.DataFrame()
all_tracks_with_scan = []   # collect per-cell tracks with scan areas
all_jumps = []              # collect per-cell jump data

for i, fn in enumerate(filenames):
    savename = os.path.splitext(fn)[0]
    locs = load_locs(dirn, fn, conv_factor)
    
    tracks = track_and_filter(
        locs,
        max_dist,
        max_off,
        exact_track_len,
        outname + savename
    )

    print(f"{savename}: tracks shape = {tracks.shape}")
    if len(tracks) < 1:
        print(f"⚠️ Skipping {savename}: no valid tracks after filtering")
        continue

    # Save per-cell tracks
    tracks.to_csv(outname + savename + ".csv", index=False)
    
    # ==========================================================
    # Jump distance & jump angle analysis
    # ==========================================================
    jumps = compute_jump_distances_and_angles(tracks)
    jumps['cell_id'] = savename
    jumps['condition'] = condition

    jumps.to_csv(outname + savename + "_jumps.csv", index=False)
    all_jumps.append(jumps)

    # ==========================================================
    # Scan areas
    # ==========================================================
    tracks_with_scan = calculate_scan_areas_and_plot(
        tracks,
        cell_id=savename,
        fn_prefix=outname + savename
    )
    all_tracks_with_scan.append(tracks_with_scan)
    
    # ==========================================================
    # iMSDs
    # ==========================================================
    ims = calc_imsds(
        tracks,
        mpp=1,
        fps=fps,
        maxlag=max_lag,
        fn=outname + savename + '_iMSDs.svg'
    )

    ims_out = ims.copy()
    ims_out.index.name = 'lag_time_s'
    ims_out.to_csv(outname + savename + "_iMSD_values.csv")

    # ==========================================================
    # Fit iMSDs
    # ==========================================================
    params = fit_imsds(
        ims,
        t_exp=0.05,
        fn=outname + savename + '_fitMSDs.svg',
        fitback=fitback
    )

    params['cell_id'] = savename
    params['track_index'] = np.arange(len(params))
    params.to_csv(outname + savename + "_fit_params.csv", index=False)

    all_params = pd.concat([all_params, params], ignore_index=True)
    all_imsds = pd.concat([all_imsds, ims], axis=1)

# ==========================================================
# combine and save jump data
# ==========================================================
all_jumps_df = pd.concat(all_jumps, ignore_index=True)
all_jumps_df.to_csv(outname + condition + "_ALL_jumps.csv", index=False)
print(f"Saved combined jump CSV: {outname + condition + '_ALL_jumps.csv'}")

# ==========================================================
# combine scan-area tracks
# ==========================================================
combined_tracks_with_scan = pd.concat(all_tracks_with_scan, ignore_index=True)
combined_csv = outname + condition + "_all_cells_scan_areas.csv"
combined_tracks_with_scan.to_csv(combined_csv, index=False)
print(f"Saved combined scan areas CSV with full localization info: {combined_csv}")

# ==========================================================
# SAVE COMBINED FIT PARAMETERS
# ==========================================================
all_params.to_csv(outname + condition + "_ALL_fit_params.csv", index=False)

# ==========================================================
# SAVE COMBINED MSD VALUES
# ==========================================================
all_imsds.index.name = "lag_time_s"
all_imsds.to_csv(outname + condition + "_ALL_iMSD_values.csv")

# ==========================================================
# Filter parameters and plot violins
# ==========================================================
all_params = all_params[(all_params['alpha'] > 0.05) & (all_params['alpha'] < 1.95)]
plot_violins(all_params, outname + condition + '_violins.svg')

# ==========================================================
# Plot combined scan areas
# ==========================================================
if all_tracks_with_scan:
    plot_all_cells_scan_areas_with_csv(
        combined_tracks_with_scan,
        combined_tracks_with_scan['cell_id'].unique(),
        fn_prefix=outname + condition
    )

combined_tracks_with_scan.to_csv(combined_csv, index=False)

# ==========================================================
# MSD summary plots
# ==========================================================
msd_data = all_imsds
msd_mean = msd_data.mean(axis=1)

# --- Linear x, log y ---
fig, ax = plt.subplots(figsize=(7,7))
ax.plot(msd_data.index, msd_data, 'k-', alpha=0.05)
ax.plot(msd_data.index, msd_mean, 'k-', alpha=1, linewidth=2)

ax.set(xlabel='lag time $t$ [s]', ylabel='MSD [$\mu$m$^2$]')
ax.set_xscale('linear')
ax.set_yscale('log')

lag_min = msd_data.index.min()
lag_max = msd_data.index.max()
lag_ticks = np.arange(50, lag_max + 50, 50)
ax.set_xticks(lag_ticks)

plt.grid(False)
plt.tight_layout()
plt.savefig(outname + condition + '_msd_linearx_logy.svg', bbox_inches='tight')
plt.close()

# --- Log-log ---
fig, ax = plt.subplots(figsize=(7,7))
ax.plot(msd_data.index, msd_data, 'k-', alpha=0.05)
ax.plot(msd_data.index, msd_mean, 'k-', alpha=1, linewidth=2)

ax.set(xlabel='lag time $t$ [s]', ylabel='MSD [$\mu$m$^2$]')
ax.set_xscale('log')
ax.set_yscale('log')

lag_ticks = [50, 100, 200, 400]
lag_ticks = [t for t in lag_ticks if lag_min <= t <= lag_max]
ax.set_xticks(lag_ticks)
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())
ax.tick_params(axis='x', rotation=0)

plt.grid(False)
plt.tight_layout()
plt.savefig(outname + condition + '_msd_loglog.svg', bbox_inches='tight')
plt.close()


In [ ]:
import os
import pandas as pd
import statsmodels.formula.api as smf

# ==========================================================
# Paths
# ==========================================================
conditions = {
    "UCM13207": r"...", #Add condition type
    "Lonafarnib": r"..." #Add condition type
}
output_dir = r"..."
os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# Load and pool all jumps, then aggregate per cell (median)
# ==========================================================
all_cell_medians = []

for cond_name, input_dir in conditions.items():
    jump_files = [f for f in os.listdir(input_dir) if f.endswith("_ALL_jumps.csv")]
    if not jump_files:
        print(f"No jump CSV files found for {cond_name}")
        continue

    for i, f in enumerate(sorted(jump_files)):
        df = pd.read_csv(os.path.join(input_dir, f))
        df['replicate'] = f"Rep{i+1}"
        df['condition'] = cond_name

        # Aggregate per cell (median jump distance)
        cell_medians = df.groupby('cell_id')['jump_um'].median().reset_index()
        cell_medians['replicate'] = f"Rep{i+1}"
        cell_medians['condition'] = cond_name
        all_cell_medians.append(cell_medians)

combined_cell_df = pd.concat(all_cell_medians, ignore_index=True)
combined_cell_df.rename(columns={'jump_um': 'median_jump_um'}, inplace=True)

print(f"Total cells loaded: {len(combined_cell_df)}")
print(f"Columns: {combined_cell_df.columns.tolist()}")

# ==========================================================
# Linear Mixed Effects Model on per-cell medians
# ==========================================================
# Model: median_jump_um ~ condition + (1|replicate)
model = smf.mixedlm("median_jump_um ~ condition",
                    data=combined_cell_df,
                    groups=combined_cell_df["replicate"])
result = model.fit()
print("\n" + "="*60)
print("Linear Mixed-Effects Model for Per-Cell Median Jump Distance")
print("="*60)

# --------------------------
# Print summary normally
# --------------------------
print(result.summary())

# --------------------------
# Print coefficients with full precision
# --------------------------
print("\nFull precision LMM results:")
print("-"*70)
print(f"{'Parameter':<30}{'Coef':>12}{'Std.Err':>12}{'z':>12}{'P>|z|':>15}")
print("-"*70)

for name, coef, se, zval, pval in zip(
        result.params.index,
        result.params.values,
        result.bse.values,
        result.tvalues.values,
        result.pvalues.values):
    print(f"{name:<30}{coef:12.6f}{se:12.6f}{zval:12.6f}{pval:15.3e}")

# ==========================================================
# save combined per-cell CSV
# ==========================================================
combined_cell_df.to_csv(os.path.join(output_dir, "ALL_CELLS_median_jumps_for_LMM.csv"), index=False)
print(f"\nCombined per-cell medians saved to {os.path.join(output_dir, 'ALL_CELLS_median_jumps_for_LMM.csv')}")


### Batch effect check using linear mixed effects model (per cell median angles level)

In [ ]:
##### import os
import pandas as pd
import statsmodels.formula.api as smf

# ==========================================================
# Paths
# ==========================================================
conditions = {
    "Controls": r"...", #Add condition type
    "C75": r"..." #Add condition type
}

output_dir = r"..."
os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# Load and pool all jumps, aggregate per cell (median angle)
# ==========================================================
all_cell_medians = []

for cond_name, input_dir in conditions.items():
    jump_files = [f for f in os.listdir(input_dir) if f.endswith("_ALL_jumps.csv")]
    if not jump_files:
        print(f"No jump CSV files found for {cond_name}")
        continue

    for i, fn in enumerate(sorted(jump_files)):
        df = pd.read_csv(os.path.join(input_dir, fn))

        # Add metadata
        replicate_name = f"Rep{i+1}"
        df["replicate"] = replicate_name
        df["condition"] = cond_name

        # ------------------------------------------------------
        # Aggregate per cell: median jump angle
        # ------------------------------------------------------
        cell_medians = (
            df.groupby("cell_id")["angle_deg"]
            .median()
            .reset_index()
        )
        cell_medians["replicate"] = replicate_name
        cell_medians["condition"] = cond_name

        all_cell_medians.append(cell_medians)

combined_cell_df = pd.concat(all_cell_medians, ignore_index=True)
combined_cell_df.rename(columns={"angle_deg": "median_jump_angle_deg"}, inplace=True)

print(f"Total cells loaded: {len(combined_cell_df)}")
print("Cells per condition:")
print(combined_cell_df["condition"].value_counts())
print(f"Columns: {combined_cell_df.columns.tolist()}")

# ==========================================================
# Linear Mixed Effects Model
# Model: median jump angle ~ condition + (1 | replicate)
# ==========================================================
model = smf.mixedlm(
    "median_jump_angle_deg ~ condition",
    data=combined_cell_df,
    groups=combined_cell_df["replicate"]
)

result = model.fit()

print("\n" + "=" * 70)
print("Linear Mixed-Effects Model: Per-Cell Median Jump Angle")
print("=" * 70)

# --------------------------
# Standard statsmodels summary
# --------------------------
print(result.summary())

# --------------------------
# Full precision output
# --------------------------
print("\nFull precision LMM results:")
print("-" * 80)
print(f"{'Parameter':<35}{'Coef':>12}{'Std.Err':>12}{'z':>12}{'P>|z|':>15}")
print("-" * 80)

for name, coef, se, zval, pval in zip(
        result.params.index,
        result.params.values,
        result.bse.values,
        result.tvalues.values,
        result.pvalues.values):
    print(f"{name:<35}{coef:12.6f}{se:12.6f}{zval:12.6f}{pval:15.3e}")

# ==========================================================
# Save per-cell table used for statistics
# ==========================================================
out_csv = os.path.join(output_dir, "ALL_CELLS_median_jump_angles_for_LMM.csv")
combined_cell_df.to_csv(out_csv, index=False)
print(f"\nPer-cell median jump angles saved to:\n{out_csv}")


### Batch effect check using Kruskal-Wallis (median telomere jumps level)

In [ ]:
import os
import pandas as pd
from scipy.stats import kruskal

# -----------------------------
# Paths for both conditions
# -----------------------------
conditions = {
    "C75": r"...", #Add condition type
    "2uM C75": r"..." #Add condition type
}
output_dir = r"..."
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# Function to run batch effect check (Kruskal-Wallis on per-cell medians)
# -----------------------------
def run_jump_batch_kruskal_per_cell(input_dir, condition_name=""):
    print("\n" + "="*60)
    print(f"Batch Kruskal-Wallis (per-cell medians) for {condition_name}")
    print("="*60)

    # Load all replicate jump CSVs
    jump_files = [f for f in os.listdir(input_dir) if f.endswith("_ALL_jumps.csv")]
    if not jump_files:
        print(f"No jump CSV files found for {condition_name}")
        return

    per_cell_dfs = []

    for i, f in enumerate(sorted(jump_files)):
        df = pd.read_csv(os.path.join(input_dir, f))
        df['Replicate'] = f"Rep{i+1}"

        # Compute median per cell for this replicate
        cell_medians = df.groupby('cell_id')[['jump_um', 'angle_deg']].median().reset_index()
        cell_medians['Replicate'] = f"Rep{i+1}"
        per_cell_dfs.append(cell_medians)

    combined = pd.concat(per_cell_dfs, ignore_index=True)

    # Run Kruskal-Wallis for jump distance and jump angle (per-cell medians)
    for metric in ['jump_um', 'angle_deg']:
        groups = [combined.loc[combined['Replicate'] == f"Rep{i+1}", metric]
                  for i in range(len(jump_files))]
        H_stat, p_val = kruskal(*groups)
        print(f"{metric} (per-cell median): H = {H_stat:.3f}, p = {p_val:.3e}")

# -----------------------------
# Run Kruskal-Wallis for each condition
# -----------------------------
for cond_name, input_dir in conditions.items():
    run_jump_batch_kruskal_per_cell(input_dir, cond_name)


### Jump Distance Analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ==========================================================
# plotting style
# ==========================================================
plt.rcParams['text.usetex'] = False
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
plt.rc('legend', fontsize=14, title_fontsize=14)
plt.style.use('seaborn-v0_8-white')
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['xtick.major.size'] = 5
plt.rcParams["font.family"] = 'Arial'

# ==========================================================
# Paths
# ==========================================================
#input_dir = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260116 Jump Distance Analysis\Controls"
#output_dir = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260116 Jump Distance Analysis\Controls\Output"

input_dir = r"..."
output_dir = r"..."

os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# Load replicate CSVs
# ==========================================================
jump_files = [f for f in os.listdir(input_dir) if f.endswith("_ALL_jumps.csv")]
print("Found jump files:", jump_files)

all_jumps = []
all_cell_metrics = []


# ==========================================================
# Per-replicate processing
# ==========================================================
for fn in jump_files:
    replicate_name = os.path.splitext(fn)[0]
    csv_path = os.path.join(input_dir, fn)
    jumps = pd.read_csv(csv_path)

    required_cols = {'jump_um', 'angle_deg', 'cell_id'}
    if not required_cols.issubset(jumps.columns):
        print(f"⚠️ Skipping {fn}: missing required columns")
        continue

    jumps['replicate'] = replicate_name
    all_jumps.append(jumps)

    # ------------------------------------------------------
    # Jump distance histogram (per replicate)
    # ------------------------------------------------------
    jump_dist = jumps['jump_um'].dropna()
    median_jump = np.median(jump_dist)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.hist(jump_dist, bins=50, density=True, alpha=0.8)
    ax.axvline(median_jump, linestyle='--', linewidth=2,
               label=f"Median = {median_jump:.3f} µm")

    ax.set_xlabel("Jump distance (µm)")
    ax.set_ylabel("Probability density")
    ax.set_title(f"{replicate_name}\nJump Distance Distribution")
    ax.legend()
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir,
                f"{replicate_name}_jump_distance_hist.svg"),
                bbox_inches="tight")
    plt.close()

    # ------------------------------------------------------
    # Jump angle histogram (per replicate)
    # ------------------------------------------------------
    jump_angle = jumps['angle_deg'].dropna()
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.hist(jump_angle, bins=36, range=(0, 180),
            density=True, alpha=0.8)

    ax.set_xlabel("Jump angle (degrees)")
    ax.set_ylabel("Probability density")
    ax.set_title(f"{replicate_name}\nJump Angle Distribution")
    ax.set_xlim(0, 180)
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir,
                f"{replicate_name}_jump_angle_hist.svg"),
                bbox_inches="tight")
    plt.close()

    # ------------------------------------------------------
    # Per-cell persistance metrics
    # ------------------------------------------------------
    for cell_id, g in jumps.groupby('cell_id'):
        angles = np.deg2rad(g['angle_deg'].dropna())
        if len(angles) < 10:
            continue

        print(f"{cond_name}: kept {len(all_cell_metrics)} cells")
        
        mean_cos = np.mean(np.cos(angles))

        all_cell_metrics.append({
            'cell_id': cell_id,
            'replicate': replicate_name,
            'mean_cos_theta': mean_cos,
            'n_jumps': len(angles)
        })

    print(f"{cond_name} | {replicate_name}: kept {len(all_cell_metrics)} cells so far")
    print(f"✓ Processed {replicate_name}")

# ==========================================================
# Pool all jumps
# ==========================================================
all_jumps_df = pd.concat(all_jumps, ignore_index=True)
cell_metrics_df = pd.DataFrame(all_cell_metrics)

cell_metrics_df.to_csv(
    os.path.join(output_dir, "HGPS_all_cells_persistance_metrics.csv"),
    index=False
)

# ==========================================================
# ALL-JUMPS pooled histograms
# ==========================================================
# Jump distance
jump_dist_all = all_jumps_df['jump_um'].dropna()
median_all = np.median(jump_dist_all)

fig, ax = plt.subplots(figsize=(7, 6))
ax.hist(jump_dist_all, bins=60, density=True, alpha=0.8)
ax.axvline(median_all, linestyle='--', linewidth=2,
           label=f"Median = {median_all:.3f} µm")
ax.set_xlabel("Jump distance (µm)")
ax.set_ylabel("Probability density")
ax.set_title("All Replicates Pooled\nJump Distance Distribution")
ax.legend()
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir,
            "ALL_REPLICATES_pooled_jump_distance_hist.svg"),
            bbox_inches="tight")
plt.close()

# Jump angle
jump_angle_all = all_jumps_df['angle_deg'].dropna()
fig, ax = plt.subplots(figsize=(7, 6))
ax.hist(jump_angle_all, bins=36, range=(0, 180),
        density=True, alpha=0.8)
ax.set_xlabel("Jump angle (degrees)")
ax.set_ylabel("Probability density")
ax.set_title("All Replicates Pooled\nJump Angle Distribution")
ax.set_xlim(0, 180)
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir,
            "ALL_REPLICATES_pooled_jump_angle_hist.svg"),
            bbox_inches="tight")
plt.close()

# ==========================================================
# Per-cell MEDIAN jump distance histogram
# ==========================================================
cell_dist_medians = (
    all_jumps_df
    .groupby('cell_id')['jump_um']
    .median()
    .dropna()
)
fig, ax = plt.subplots(figsize=(7, 6))
ax.hist(cell_dist_medians, bins=20, density=True, alpha=0.8)
ax.set_xlabel("Median jump distance per cell (µm)")
ax.set_ylabel("Probability density")
ax.set_title("Per-Cell Medians Pooled\nJump Distance Distribution")
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir,
            "ALL_REPLICATES_per_cell_median_jump_distance_hist.svg"),
            bbox_inches="tight")
plt.close()

# ==========================================================
# Anisotropy violin plot
# ==========================================================
fig, ax = plt.subplots(figsize=(4,6))

sns.violinplot(
    y=cell_metrics_df['mean_cos_theta'],
    inner="point",   # show the median as a line
    cut=0,
    bw=0.05,
    color="#9ecae1",
    ax=ax
)

# Overlay per-cell points
sns.stripplot(
    y=cell_metrics_df['mean_cos_theta'],
    color='k',
    size=5,
    alpha=0.7,
    ax=ax,
    jitter=True
)

ax.set_ylabel("Directional persistence ⟨cos θ⟩")
ax.set_xticks([])
ax.set_ylim(-1.05, 1.05)
ax.set_title("Telomere Jump Anisotropy\nPer-Cell Distribution", fontsize=16, fontweight='bold')
ax.grid(False)

plt.tight_layout()
plt.savefig(
    os.path.join(output_dir, "ALL_REPLICATES_per_cell_anisotropy_violin.svg"),
    bbox_inches='tight', dpi=300
)
plt.close()
print("✓ Saved per-cell anisotropy violin plot")


### Compare jump conditions back to back

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ==========================================================
# plotting style
# ==========================================================
plt.rcParams['text.usetex'] = False
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.labelsize'] = 16
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
plt.rc('legend', fontsize=14, title_fontsize=14)
plt.style.use('seaborn-v0_8-white')
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['xtick.major.size'] = 5
plt.rcParams["font.family"] = 'Arial'

# ==========================================================
# Paths for all conditions
# ==========================================================
conditions = {
    "Controls": r"...",
    "HGPS": r"...",
    "Lonafarnib": r"...",
    "UCM13207": r"...",
    "C75": r"...",
    "2 uM C75": r"...",
}
output_dir = r"..."
os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# Define plotting colors
# ==========================================================
condition_colors = {
    "Controls": "lightsteelblue",
    "HGPS": "slateblue",
    "Lonafarnib": "teal",
    "UCM13207": "mediumaquamarine",
    "C75": "mediumseagreen",
    "2 uM C75": "darkslategray"
}

# ==========================================================
# Process each condition
# ==========================================================
all_data = {}

def circular_metrics_deg(angles_deg):
    angles_rad = np.deg2rad(angles_deg)
    C = np.mean(np.cos(angles_rad))
    S = np.mean(np.sin(angles_rad))
    R = np.sqrt(C**2 + S**2)
    mean_dir = np.rad2deg(np.arctan2(S, C))
    return R, mean_dir

for cond_name, input_dir in conditions.items():
    jump_files = [f for f in os.listdir(input_dir) if f.endswith("_ALL_jumps.csv")]
    print(f"Found {cond_name} jump files:", jump_files)
    
    all_jumps = []
    all_cell_metrics = []

    for fn in jump_files:
        replicate_name = os.path.splitext(fn)[0]
        csv_path = os.path.join(input_dir, fn)
        jumps = pd.read_csv(csv_path)

        required_cols = {'jump_um', 'angle_deg', 'cell_id'}
        if not required_cols.issubset(jumps.columns):
            print(f"⚠️ Skipping {fn}: missing required columns")
            continue

        jumps['replicate'] = replicate_name
        jumps["cell_id"] = jumps["replicate"] + "_" + jumps["cell_id"].astype(str)
        all_jumps.append(jumps)

        # ------------------------------------------------------
        # Per-cell persistence metrics
        # ------------------------------------------------------
        for cell_id, g in jumps.groupby('cell_id'):
            angles_deg = g['angle_deg'].dropna()
            if len(angles_deg) < 10:
                continue
            mean_cos = np.mean(np.cos(np.deg2rad(angles_deg)))
            all_cell_metrics.append({
                'cell_id': cell_id,
                'replicate': replicate_name,
                'mean_cos_theta': mean_cos,
                'n_jumps': len(angles_deg)
            })

    all_jumps_df = pd.concat(all_jumps, ignore_index=True)
    cell_metrics_df = pd.DataFrame(all_cell_metrics)
    cell_metrics_df.to_csv(os.path.join(output_dir, f"{cond_name}_all_cells_persistence_metrics.csv"), index=False)
    
    all_data[cond_name] = {
        "jumps": all_jumps_df,
        "cell_metrics": cell_metrics_df
    }
    print(f"✓ Finished processing {cond_name}")

# ==========================================================
# Histogram: all jumps pooled (Controls first, HGPS second)
# ==========================================================
for metric, xlabel in [
    ("jump_um", "Jump distance (µm)"),
    ("angle_deg", "Consecutive-step angle (degrees)")
]:
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_order = ["Controls", "HGPS"]
    
    if metric == "angle_deg":
        bins = np.linspace(0, 180, 37)
    else:
        bins = 60
    
    max_density = 0
    for i, cond_name in enumerate(plot_order):
        values = all_data[cond_name]["jumps"][metric].dropna()
        counts, _, _ = ax.hist(
            values, bins=bins, density=True, alpha=0.5,
            color=condition_colors[cond_name],
            edgecolor='black', linewidth=1.2,
            label=cond_name
        )
        max_density = max(max_density, counts.max())
        median_val = np.median(values)
        ax.axvline(median_val, color=condition_colors[cond_name], linestyle='--', linewidth=2)
        ax.text(0.98, 0.95 - i*0.05, f"{cond_name} median = {median_val:.3f}",
                color=condition_colors[cond_name], ha='right', va='top', transform=ax.transAxes, fontsize=12)

    if metric == "jump_um":
        ax.set_xlim(0, 0.5)
    else:
        ax.set_xlim(0, 180)
    ax.set_ylim(0, max_density * 1.15)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Probability density")
    ax.set_title(f"{metric} distribution: Controls vs HGPS")
    ax.legend()
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{metric}_comparison_hist.svg"), bbox_inches="tight")
    plt.close()

# ==========================================================
# Histogram: all-condition pooled jumps
# ==========================================================
fig, ax = plt.subplots(figsize=(9, 7))
labels = ["Controls", "HGPS", "Lonafarnib", "UCM13207", "C75", "2 uM C75"]
bins = 60
max_density = 0
text_lines = []

for cond_name in labels:
    values = all_data[cond_name]["jumps"]["jump_um"].dropna()
    counts, _, _ = ax.hist(
        values, bins=bins, density=True, alpha=0.45,
        color=condition_colors[cond_name],
        edgecolor='black', linewidth=1.1, label=cond_name
    )
    max_density = max(max_density, counts.max())
    median_val = np.median(values)
    ax.axvline(median_val, color=condition_colors[cond_name], linestyle='--', linewidth=2)
    text_lines.append(f"{cond_name} median = {median_val:.3f}")

ax.text(0.98, 0.98, "\n".join(text_lines), transform=ax.transAxes,
        ha='right', va='top', fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
ax.set_xlim(0, 0.5)
ax.set_ylim(0, max_density * 1.2)
ax.set_xlabel("Jump distance (µm)")
ax.set_ylabel("Probability density")
ax.set_title("Jump distance distribution (all conditions)")
ax.legend()
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "jump_distance_hist_all_conditions.svg"), dpi=300)
plt.close()

# ==========================================================
# Bar chart: per-cell mean ± SEM of median jump distance
# ==========================================================
fig, ax = plt.subplots(figsize=(7, 6))
medians = []
sems = []
text_lines = []

for cond_name in labels:
    per_cell_vals = all_data[cond_name]["jumps"].groupby("cell_id")["jump_um"].median().dropna()
    mean_val = np.mean(per_cell_vals)
    sem_val = np.std(per_cell_vals, ddof=1) / np.sqrt(len(per_cell_vals))
    medians.append(mean_val)
    sems.append(sem_val)
    text_lines.append(f"{cond_name} mean = {mean_val:.3f}")
    print(f"{cond_name} (per-cell medians): n = {len(per_cell_vals)} cells")

x = np.arange(len(labels))
ax.bar(
    x, medians, yerr=sems, capsize=6,
    color=[condition_colors[l] for l in labels],
    edgecolor='black', linewidth=1.5
)
ax.set_ylim(0, max(medians[i]+sems[i] for i in range(len(labels))) * 1.35)
ax.text(0.98, 0.98, "\n".join(text_lines), transform=ax.transAxes,
        ha='right', va='top', fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel("Mean jump distance per cell (µm)")
ax.set_title("Per-cell median jump distance\nAll conditions (mean ± SEM)")
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "jump_distance_per_cell_mean_bar_all_conditions.svg"), dpi=300, bbox_inches="tight")
plt.close()

# ==========================================================
# Histogram: all-condition pooled jump angles
# ==========================================================
fig, ax = plt.subplots(figsize=(9, 7))
bins = np.linspace(0, 180, 37)
max_density = 0
text_lines = []

for cond_name in labels:
    angles = all_data[cond_name]["jumps"]["angle_deg"].dropna()
    counts, _, _ = ax.hist(
        angles, bins=bins, density=True, alpha=0.45,
        color=condition_colors[cond_name],
        edgecolor='black', linewidth=1.1, label=cond_name
    )
    max_density = max(max_density, counts.max())
    median_angle = np.median(angles)
    ax.axvline(median_angle, color=condition_colors[cond_name], linestyle='--', linewidth=2)
    text_lines.append(f"{cond_name} median = {median_angle:.1f}°")

ax.text(0.98, 0.98, "\n".join(text_lines), transform=ax.transAxes,
        ha='right', va='top', fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
ax.set_xlim(0, 180)
ax.set_ylim(0, max_density * 1.2)
ax.set_xlabel("Consecutive-step angle (degrees)")
ax.set_ylabel("Probability density")
ax.set_title("Jump angle distribution (all conditions)")
ax.legend()
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "jump_angle_hist_all_conditions.svg"), dpi=300)
plt.close()

# ==========================================================
# Bar chart: per-cell mean ± SEM of median jump angle
# ==========================================================
fig, ax = plt.subplots(figsize=(7, 6))
medians = []
sems = []
text_lines = []

for cond_name in labels:
    per_cell_vals = all_data[cond_name]["jumps"].groupby("cell_id")["angle_deg"].median().dropna()
    mean_val = np.mean(per_cell_vals)
    sem_val = np.std(per_cell_vals, ddof=1) / np.sqrt(len(per_cell_vals))
    medians.append(mean_val)
    sems.append(sem_val)
    text_lines.append(f"{cond_name} mean = {mean_val:.3f}")
    print(f"{cond_name} (per-cell median angles): n = {len(per_cell_vals)} cells")

x = np.arange(len(labels))
ax.bar(
    x, medians, yerr=sems, capsize=6,
    color=[condition_colors[l] for l in labels],
    edgecolor='black', linewidth=1.5
)
ax.set_ylim(0, max(medians[i]+sems[i] for i in range(len(labels))) * 1.35)
ax.text(0.98, 0.98, "\n".join(text_lines), transform=ax.transAxes,
        ha='right', va='top', fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel("Mean jump angle per cell (degrees)")
ax.set_title("Per-cell median jump angle\nAll conditions (mean ± SEM)")
ax.grid(False)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "jump_angle_per_cell_mean_bar_all_conditions.svg"), dpi=300, bbox_inches="tight")
plt.close()

# ==========================================================
# Polar Rose Plots: 0-180 data mirrored
# ==========================================================

# Use the same order for the "all conditions" plot
labels = ["Controls", "HGPS", "Lonafarnib", "UCM13207", "C75", "2 uM C75"]

def plot_polar_rose(cond_list, title_suffix, filename):
    fig = plt.figure(figsize=(6 * len(cond_list), 6))
    
    for i, cond_name in enumerate(cond_list):
        ax = fig.add_subplot(1, len(cond_list), i + 1, projection='polar')
        
        # Pull data from all_data dictionary
        angles_0_180 = all_data[cond_name]["jumps"]["angle_deg"].dropna().values
        
        # Mirror the 0-180 data to -180 to 0 to create the full rose
        mirrored_angles = -angles_0_180
        full_angles_360 = np.concatenate([angles_0_180, mirrored_angles])
        angles_rad = np.deg2rad(full_angles_360)
        
        # Setup bins (20 degree bins)
        number_of_bins = 18 
        bin_edges = np.linspace(-np.pi, np.pi, number_of_bins + 1)
        counts, edges = np.histogram(angles_rad, bins=bin_edges, density=True)
        
        # Plot bars
        width = (2 * np.pi) / number_of_bins
        ax.bar(edges[:-1], counts, width=width, bottom=0.0, 
               color=condition_colors[cond_name], edgecolor='black', alpha=0.7)
        
        # Formatting
        ax.set_theta_zero_location('E')  # 0 degrees at East (right)
        ax.set_theta_direction(1)        # Counter-clockwise
        ax.set_title(f"{cond_name}", pad=20)
        
        # ticks to reflect 0-180 logic
        ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
        ax.set_xticklabels(['0°', '45°', '90°', '135°', '180°', '135°', '90°', '45°'])

    plt.suptitle(f"Angular Jump Distributions: {title_suffix}", fontsize=22, y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), bbox_inches="tight", dpi=300)
    print(f"✓ Saved polar plot: {filename}")
    plt.show()

# --- UPDATED OVERLAID PLOT: FILLED WITH DARK OUTLINES ---
def plot_overlaid_polar_rose(cond_list, title, filename):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='polar')
    
    number_of_bins = 18
    bin_edges = np.linspace(-np.pi, np.pi, number_of_bins + 1)
    width = (2 * np.pi) / number_of_bins

    for cond_name in cond_list:
        angles_0_180 = all_data[cond_name]["jumps"]["angle_deg"].dropna().values
        mirrored_angles = -angles_0_180
        full_angles_360 = np.concatenate([angles_0_180, mirrored_angles])
        angles_rad = np.deg2rad(full_angles_360)
        
        counts, edges = np.histogram(angles_rad, bins=bin_edges, density=True)
        
        # base color for the condition
        base_color = mpl.colors.to_rgb(condition_colors[cond_name])
        
        # semi-transparent version for the fill
        fill_color = (*base_color, 0.2)  # 0.3 is the fill alpha
        
        # Plot bars: use the fill_color for the face, but edgecolor solid (alpha=1)
        ax.bar(edges[:-1], counts, width=width, bottom=0.0, 
               facecolor=fill_color, 
               edgecolor=condition_colors[cond_name], 
               linewidth=2, 
               label=cond_name)

    # Formatting 
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
    ax.set_xticklabels(['0°', '45°', '90°', '135°', '180°', '135°', '90°', '45°'])
    ax.set_title(title, fontsize=20, pad=30)
    
    # Place legend outside the plot
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), bbox_inches="tight", dpi=300)
    print(f"✓ Saved overlaid dark-outline polar plot: {filename}")
    plt.show()

# comparison figures
plot_polar_rose(["Controls", "HGPS"], "Controls vs HGPS", "polar_jump_angle_comparison.svg")
plot_polar_rose(labels, "Individual Conditions", "polar_jump_angle_individual.svg")

# overlaid comparison
plot_overlaid_polar_rose(labels, "Overlaid Jump Distributions (All Conditions)", "polar_jump_angle_OVERLAID.svg")

In [ ]:
print("\nCorrected jump and angle counts (all conditions):")

for cond_name, cond_data in all_data.items():
    jumps_df = cond_data["jumps"].copy()

    # Number of jumps = number of rows
    n_jumps = len(jumps_df)

    # Number of tracks (cells)
    n_tracks = jumps_df["cell_id"].nunique()

    # True number of angles = sum over tracks of (n_jumps_per_track - 1)
    # which is equivalent to total_jumps - n_tracks
    n_angles_true = n_jumps - n_tracks

    print(
        f"{cond_name}: "
        f"{n_jumps} jumps, "
        f"{n_angles_true} angles (corrected), "
        f"{n_tracks} cells"
    )



### Polar Rose Plot for Jump Angles

In [ ]:
# ==========================================================
# Polar Rose Plots: 0-180 data mirrored
# ==========================================================

# Use the same order for the "all conditions" plot
labels = ["Controls", "HGPS", "Lonafarnib", "UCM13207", "C75", "2 uM C75"]

def plot_polar_rose(cond_list, title_suffix, filename):
    fig = plt.figure(figsize=(6 * len(cond_list), 6))
    
    for i, cond_name in enumerate(cond_list):
        ax = fig.add_subplot(1, len(cond_list), i + 1, projection='polar')
        
        # Pull data from all_data dictionary
        angles_0_180 = all_data[cond_name]["jumps"]["angle_deg"].dropna().values
        
        # Mirror the 0-180 data to -180 to 0 to create the full rose
        mirrored_angles = -angles_0_180
        full_angles_360 = np.concatenate([angles_0_180, mirrored_angles])
        angles_rad = np.deg2rad(full_angles_360)
        
        # Setup bins (20 degree bins)
        number_of_bins = 18 
        bin_edges = np.linspace(-np.pi, np.pi, number_of_bins + 1)
        counts, edges = np.histogram(angles_rad, bins=bin_edges, density=True)
        
        # Plot bars
        width = (2 * np.pi) / number_of_bins
        ax.bar(edges[:-1], counts, width=width, bottom=0.0, 
               color=condition_colors[cond_name], edgecolor='black', alpha=0.7)
        
        # Formatting
        ax.set_theta_zero_location('E')  # 0 degrees at East (right)
        ax.set_theta_direction(1)        # Counter-clockwise
        ax.set_title(f"{cond_name}", pad=20)
        
        # Customize ticks to reflect 0-180 logic
        ax.set_xticks(np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315]))
        ax.set_xticklabels(['0°', '45°', '90°', '135°', '180°', '135°', '90°', '45°'])

    plt.suptitle(f"Angular Jump Distributions: {title_suffix}", fontsize=22, y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), bbox_inches="tight", dpi=300)
    print(f"✓ Saved polar plot: {filename}")
    plt.show()

# two comparison figures
plot_polar_rose(["Controls", "HGPS"], "Controls vs HGPS", "polar_jump_angle_comparison.svg")
plot_polar_rose(labels, "All Conditions", "polar_jump_angle_all_conditions.svg")

### Overlay tracks onto microscopy images
> input the csv and one frame of the tiff stack

In [ ]:
# -------------------------------------------------------------
#Overlay Telomere Tracks on Microscopy Image (CSV in um, with particle column (output from previous sections)
#Code flips tracks overy y axis so that the x axis can read from left to right (0 - 10 um), so remember to flip
#the microscopy image you are using vertically first
# -------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from skimage import io
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.spatial import ConvexHull

def plot_tracks_on_image(
    image_path,
    csv_path,
    pixel_size_um,
    output_path,
    cmap_name='viridis',
    plot_trajectories=True,
    connect_lines=True
):
    """
    Overlay telomere tracks on a microscopy image.
    
    Saves two images:
    1) Tracks colored by scan area
    2) Tracks colored by time (frame number)
    
    Parameters:
    -----------
    image_path : str
        Path to the 2D microscopy image (single frame or projection).
    csv_path : str
        CSV with tracked telomere coordinates (columns: frame, x, y, particle).
        Units: µm
    pixel_size_um : float
        Size of one pixel in µm.
    output_path : str
        Base file path to save the overlay images.
        Scan area: output_path + '_scan_area.png'
        Time: output_path + '_time.png'
    cmap_name : str
        Name of matplotlib colormap to color trajectories.
    plot_trajectories : bool
        If True, draw lines for each particle. If False, scatter points only.
    connect_lines : bool
        If True, connect points of same particle with lines.
    """

    # --- Load image ---
    img = io.imread(image_path)
    if img.ndim > 2:
        img = np.max(img, axis=0)
    
    # --- Load tracked CSV ---
    df = pd.read_csv(csv_path)
    required_cols = ['frame', 'x', 'y', 'particle']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must contain column '{col}'")
    
    # --- Compute per-particle scan areas ---
    scan_areas = {}
    for pid in df['particle'].unique():
        single = df[df['particle'] == pid]
        if len(single) > 2:
            hull = ConvexHull(single[['x', 'y']].values)
            scan_areas[pid] = hull.volume  # µm²
        else:
            scan_areas[pid] = 0.0
    areas = np.array(list(scan_areas.values()))
    
    # --- 1) Plot tracks colored by scan area ---
    vmin, vmax = np.percentile(areas, [5, 95])
    norm_area = Normalize(vmin=vmin, vmax=vmax)
    cmap_area = cm.get_cmap(cmap_name)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, cmap='gray', origin='upper',
              extent=[0, img.shape[1]*pixel_size_um, 0, img.shape[0]*pixel_size_um])
    
    for pid in df['particle'].unique():
        single = df[df['particle'] == pid].sort_values('frame')
        color = cmap_area(norm_area(scan_areas[pid]))
        x = single['x'].values
        y = single['y'].values
        if plot_trajectories and connect_lines:
            ax.plot(x, y, color=color, linewidth=1.5, alpha=0.9)
        else:
            ax.scatter(x, y, color=color, s=20, alpha=0.8)
    
    ax.set_xlabel("x [µm]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.set_title("Telomere Tracks Overlay (Scan Area)", fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    
    sm = cm.ScalarMappable(cmap=cmap_area, norm=norm_area)
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Scan area (µm²)', fontsize=14)
    
    plt.tight_layout()
    plt.savefig(output_path + '_scan_area.png', dpi=300)
    plt.close()
    
    # --- 2) Plot tracks colored by time (Minutes) ---
    frame_interval_s = 30  # seconds per frame
    times = df['frame'].values * frame_interval_s / 60  # convert to minutes
    vmin_time, vmax_time = times.min(), times.max()
    norm_time = Normalize(vmin=vmin_time, vmax=vmax_time)
    cmap_time = cm.get_cmap(cmap_name)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, cmap='gray', origin='upper',
              extent=[0, img.shape[1]*pixel_size_um, 0, img.shape[0]*pixel_size_um])
    
    for pid in df['particle'].unique():
        single = df[df['particle'] == pid].sort_values('frame')
        x = single['x'].values
        y = single['y'].values
        times_particle = single['frame'].values * frame_interval_s / 60 # convert to minutes
        colors = cmap_time(norm_time(times_particle)) # generate colors
        if plot_trajectories and connect_lines:
            for i in range(len(x)-1):
                ax.plot(x[i:i+2], y[i:i+2], color=colors[i], linewidth=1.5, alpha=0.9)
        else:
            ax.scatter(x, y, color=colors, s=20, alpha=0.8)
    
    ax.set_xlabel("x [µm]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.set_title("Telomere Tracks Overlay (Time)", fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    
    sm = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Time (mins)', fontsize=14)  # or 'Time (min)' if you divide by 60

    plt.tight_layout()
    plt.savefig(output_path + '_time.png', dpi=300)
    plt.close()


In [ ]:
plot_tracks_on_image(
    image_path="...",
    csv_path="...",
    pixel_size_um=0.1307,       # microscope calibration
    output_path="...",
    cmap_name='viridis',
    plot_trajectories=True,
    connect_lines=True
)


### Inspect individual trajectory and motion over time

In [ ]:
# ============================================================
# Function: Select and plot an individual trajectory by clicking (zoomed)
# ============================================================
%matplotlib qt

def plot_single_trajectory_zoomed(csv_path, frame_interval_s=30, save_prefix=None, padding_um=1.0):
    """
    Select a single particle trajectory by clicking on a plot of all tracks,
    then plot:
      1) The trajectory colored by time (minutes), zoomed around the trajectory
      2) x(t) and y(t) vs time (minutes)

    Now also:
      - Computes scan area for this track (convex hull), writes it top-right
      - Plots transparent shaded polygon of the scan area

    Optionally saves all plots to PNG files.
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib import cm
    from matplotlib.colors import Normalize
    from matplotlib.collections import LineCollection
    from scipy.spatial import ConvexHull

    # --- Load tracked CSV ---
    df = pd.read_csv(csv_path)
    required_cols = ['frame', 'x', 'y', 'particle']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must contain column '{col}'")
    
    # --- Plot all tracks for selection ---
    fig, ax = plt.subplots(figsize=(8,8))
    for pid in df['particle'].unique():
        single_trace = df[df['particle'] == pid].sort_values('frame')
        ax.plot(single_trace['x'], single_trace['y'], alpha=0.3, color='gray')
    ax.set_title("Click near the trajectory you want to select")
    ax.set_xlabel("x [µm]"); ax.set_ylabel("y [µm]")
    ax.set_aspect('equal')
    
    coords = plt.ginput(1)  # user clicks once
    plt.close()
    
    if len(coords) == 0:
        print("No point selected. Exiting.")
        return
    
    clicked_x, clicked_y = coords[0]
    
    # --- Find the particle whose trajectory has the closest point to click ---
    distances = df.groupby('particle').apply(
        lambda g: np.min(np.sqrt((g['x']-clicked_x)**2 + (g['y']-clicked_y)**2))
    )
    selected_id = distances.idxmin()
    print(f"Selected particle ID: {selected_id}")
    
    # --- Extract the single trajectory ---
    single = df[df['particle'] == selected_id].sort_values('frame')
    times = single['frame'].values * frame_interval_s / 60  # minutes

    # ============================================================
    # Compute scan area (convex hull)
    # ============================================================
    points = single[['x', 'y']].values
    if len(points) >= 3:
        hull = ConvexHull(points)
        scan_area = hull.volume  #2D hull area
        hull_vertices = points[hull.vertices]
    else:
        scan_area = 0
        hull_vertices = None

    # --- 1) Trajectory colored by time (zoomed) ---
    norm_time = Normalize(vmin=times.min(), vmax=times.max())
    cmap_time = cm.viridis
    colors = cmap_time(norm_time(times))
    
    fig, ax = plt.subplots(figsize=(6,6))

    # ============================================================
    # Plot transparent convex hull shading
    # ============================================================
    if hull_vertices is not None:
        ax.fill(
            hull_vertices[:,0],
            hull_vertices[:,1],
            color='gray',
            alpha=0.15,
            linewidth=0
        )

    # --- plot trajectory ---
    for i in range(len(single)-1):
        ax.plot(single['x'].values[i:i+2], single['y'].values[i:i+2], color=colors[i], linewidth=2)
    
    # Set zoomed limits
    # Make the box limits square
    x_min, x_max = single['x'].min(), single['x'].max()
    y_min, y_max = single['y'].min(), single['y'].max()

    x_center = 0.5 * (x_min + x_max)
    y_center = 0.5 * (y_min + y_max)

    x_range = x_max - x_min
    y_range = y_max - y_min

    base_size = max(x_range, y_range)

    # Use a tighter padding fraction (15%)
    padding_fraction = 0.15  # adjust to taste
    half_size = 0.5 * base_size * (1 + padding_fraction)

    # Apply square limits
    ax.set_xlim(x_center - half_size, x_center + half_size)
    ax.set_ylim(y_center - half_size, y_center + half_size)

    ax.set_aspect('equal')
    
    ax.set_xlabel("x [µm]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.set_title(f"Particle {selected_id} Trajectory (Time, mins)", fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    
    sm = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("Time (mins)", fontsize=12)

    # ============================================================
    # Add scan area text annotation in top-right corner
    # ============================================================
    ax.text(
        0.98, 0.98,
        f"Scan area = {scan_area:.3f} µm²",
        ha='right', va='top',
        transform=ax.transAxes,
        fontsize=12,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none')
    )

    plt.tight_layout()
    
    if save_prefix is not None:
        fig.savefig(save_prefix + "_trajectory.svg", dpi=300)
    

    # ----------------- 2) x(t) and y(t) vs time -------------------
    x = single['x'].values
    y = single['y'].values
    
    norm = Normalize(vmin=times.min(), vmax=times.max())
    cmap = cm.viridis
    colors = cmap(norm(times))
    
    # --- x vs t ---
    fig, ax = plt.subplots(figsize=(6,4))
    for i in range(len(times)-1):
        ax.plot(times[i:i+2], x[i:i+2], color=colors[i], linewidth=2)
    ax.scatter(times, x, color=colors, s=15, alpha=0.9)
    ax.set_xlabel("Time [min]", fontsize=14)
    ax.set_ylabel("x [µm]", fontsize=14)
    ax.set_title(f"Particle {selected_id} - x vs t", fontsize=16, fontweight='bold')
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Time [min]', fontsize=12)
    ax.grid(False)
    plt.tight_layout()
    if save_prefix is not None:
        fig.savefig(save_prefix + "_x_vs_t.svg", dpi=300)
    
    # --- y vs t ---
    fig, ax = plt.subplots(figsize=(6,4))
    for i in range(len(times)-1):
        ax.plot(times[i:i+2], y[i:i+2], color=colors[i], linewidth=2)
    ax.scatter(times, y, color=colors, s=15, alpha=0.9)
    ax.set_xlabel("Time [min]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.set_title(f"Particle {selected_id} - y vs t", fontsize=16, fontweight='bold')
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label('Time [min]', fontsize=12)
    ax.grid(False)
    plt.tight_layout()
    if save_prefix is not None:
        fig.savefig(save_prefix + "_y_vs_t.svg", dpi=300)
    
    plt.show()


In [ ]:
plot_single_trajectory_zoomed(
    csv_path="...",
    #pixel_size_um=0.1307,       # microscope calibration    
    frame_interval_s=30,
    save_prefix="...",
    padding_um=1.0
)


### New color scheme of individual trajectory for figure 1

In [ ]:
# ============================================================
# Function: Select and plot an individual trajectory by clicking (zoomed)
# ============================================================
%matplotlib qt

def plot_single_trajectory_zoomed(csv_path, frame_interval_s=30, save_prefix=None, padding_um=1.0):
    """
    Select a single particle trajectory by clicking on a plot of all tracks,
    then plot:
      1) The trajectory colored by time (minutes), zoomed around the trajectory
      2) x(t) and y(t) vs time (minutes)

    Computes scan area for this track (convex hull) and writes it top-right
    (no hull shading plotted).
    """
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize, LinearSegmentedColormap
    from matplotlib import cm
    from scipy.spatial import ConvexHull

    # --- Load tracked CSV ---
    df = pd.read_csv(csv_path)
    required_cols = ['frame', 'x', 'y', 'particle']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must contain column '{col}'")

    # --- Plot all tracks for selection ---
    fig, ax = plt.subplots(figsize=(8, 8))
    for pid in df['particle'].unique():
        single_trace = df[df['particle'] == pid].sort_values('frame')
        ax.plot(single_trace['x'], single_trace['y'], alpha=0.3, color='gray')

    ax.set_title("Click near the trajectory you want to select")
    ax.set_xlabel("x [µm]")
    ax.set_ylabel("y [µm]")
    ax.set_aspect('equal')

    coords = plt.ginput(1)
    plt.close()

    if len(coords) == 0:
        print("No point selected. Exiting.")
        return

    clicked_x, clicked_y = coords[0]

    # --- Find closest trajectory ---
    distances = df.groupby('particle').apply(
        lambda g: np.min(np.sqrt((g['x'] - clicked_x) ** 2 + (g['y'] - clicked_y) ** 2))
    )
    selected_id = distances.idxmin()
    print(f"Selected particle ID: {selected_id}")

    # --- Extract trajectory ---
    single = df[df['particle'] == selected_id].sort_values('frame')
    times = single['frame'].values * frame_interval_s / 60  # minutes

    # --- Compute scan area (convex hull, no shading) ---
    points = single[['x', 'y']].values
    if len(points) >= 3:
        hull = ConvexHull(points)
        scan_area = hull.volume  # 2D area
    else:
        scan_area = 0

    # ============================================================
    # Colormap: viridis
    # ============================================================
    from matplotlib import cm
    from matplotlib.colors import Normalize

    cmap_time = cm.viridis
    norm_time = Normalize(vmin=times.min(), vmax=times.max())
    colors = cmap_time(norm_time(times))



    # --- 1) Trajectory plot ---
    fig, ax = plt.subplots(figsize=(6, 6))

    for i in range(len(single) - 1):
        ax.plot(
            single['x'].values[i:i+2],
            single['y'].values[i:i+2],
            color=colors[i],
            linewidth=2
        )

    # --- Zoomed square limits ---
    x_min, x_max = single['x'].min(), single['x'].max()
    y_min, y_max = single['y'].min(), single['y'].max()

    x_center = 0.5 * (x_min + x_max)
    y_center = 0.5 * (y_min + y_max)
    base_size = max(x_max - x_min, y_max - y_min)
    padding_fraction = 0.15
    half_size = 0.5 * base_size * (1 + padding_fraction)

    ax.set_xlim(x_center - half_size, x_center + half_size)
    ax.set_ylim(y_center - half_size, y_center + half_size)
    ax.set_aspect('equal')

    ax.set_xlabel("x [µm]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.set_title(
        f"Particle {selected_id} Trajectory (Time, mins)",
        fontsize=16,
        fontweight='bold'
    )

    sm = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("Time (mins)", fontsize=12)

    plt.tight_layout()

    if save_prefix is not None:
        fig.savefig(save_prefix + "_trajectory.svg", dpi=300)

    # ----------------- 2) x(t) vs time -------------------
    x = single['x'].values
    y = single['y'].values

    fig, ax = plt.subplots(figsize=(6, 4))
    for i in range(len(times) - 1):
        ax.plot(times[i:i+2], x[i:i+2], color=colors[i], linewidth=2)
    ax.scatter(times, x, color=colors, s=15)

    ax.set_xlabel("Time [min]", fontsize=14)
    ax.set_ylabel("x [µm]", fontsize=14)
    ax.set_title(f"Particle {selected_id} - x vs t", fontsize=16, fontweight='bold')

    sm = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
    sm.set_array(times)
    plt.colorbar(sm, ax=ax).set_label("Time [min]", fontsize=12)

    plt.tight_layout()
    if save_prefix is not None:
        fig.savefig(save_prefix + "_x_vs_t.png", dpi=300)

    # ----------------- 3) y(t) vs time -------------------
    fig, ax = plt.subplots(figsize=(6, 4))
    for i in range(len(times) - 1):
        ax.plot(times[i:i+2], y[i:i+2], color=colors[i], linewidth=2)
    ax.scatter(times, y, color=colors, s=15)

    ax.set_xlabel("Time [min]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.set_title(f"Particle {selected_id} - y vs t", fontsize=16, fontweight='bold')

    sm = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
    sm.set_array([])
    plt.colorbar(sm, ax=ax).set_label("Time [min]", fontsize=12)

    plt.tight_layout()
    if save_prefix is not None:
        fig.savefig(save_prefix + "_y_vs_t.png", dpi=300)

    plt.show()

plot_single_trajectory_zoomed(
    csv_path="...",
    #pixel_size_um=0.1307,       # microscope calibration    
    frame_interval_s=30,
    save_prefix="...",
    padding_um=1.0
)

### Plotting combined MSDs and Violin plots across replicates

In [ ]:
# ===============================================================
# COMBINE REPLICATES AND PLOT FINAL MSD + VIOLIN FIGURES
# ===============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker as mticker

# --- USER PARAMETERS ---
replicate_csvs = {
    'Progeria': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep1_HGPS_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep2_HGPS_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep3_HGPS_ALL_iMSD_values.csv'
    ],
    'Control': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep1_Controls_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep2_Controls_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep3_Controls_ALL_iMSD_values.csv'
    ]
}

fit_csvs = {
    'Progeria': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep1_HGPS_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep2_HGPS_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep3_HGPS_ALL_fit_params.csv'
    ],
    'Control': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep1_Controls_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep2_Controls_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep3_Controls_ALL_fit_params.csv'
    ]
}

output_prefix = r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Output\no backlund\60 frames\final_combined_figures'

# ==============================
# COLOR SCHEMES PER CONDITION
# ==============================
color_schemes = {
    'Progeria': ["mediumslateblue", "mediumorchid", "darkmagenta"],
    'Control': ["lightsteelblue", "steelblue", "darkslateblue"]
    #'Lonafarnib': ["cadetblue", "teal", "seagreen"],
    #'UCM-13207': ["mediumaquamarine", "mediumseagreen", "lightseagreen"]
}

# ==============================
# FUNCTION: Combine MSD replicates
# ==============================
def load_and_combine_msds(msd_files):
    replicate_msds = []
    for f in msd_files:
        df = pd.read_csv(f, index_col=0)
        replicate_msds.append(df)
    return replicate_msds

def plot_combined_msds(replicate_msds, condition_name, ax=None):
    colors = color_schemes[condition_name]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))
    
    # Compute mean MSD per replicate
    mean_msds = []
    for df in replicate_msds:
        mean_msds.append(df.mean(axis=1))
        ax.plot(df.index, df, color='gray', alpha=0.05)  # all tracks
    
    # Plot replicate means with specified colors
    for i, median_df in enumerate(mean_msds):
        ax.plot(median_df.index, median_df, color=colors[i], linewidth=2, label=f'Replicate {i+1}')
    
    # Compute mean ± SEM across replicates
    combined = pd.concat(mean_msds, axis=1)
    combined_mean = combined.mean(axis=1)
    combined_sem = combined.sem(axis=1)
    
    ax.plot(combined_mean.index, combined_mean, color='k', linewidth=3, label='Mean ± SEM')
    ax.fill_between(combined_mean.index,
                    combined_mean - combined_sem,
                    combined_mean + combined_sem,
                    color='k', alpha=0.35)
    
    # Log-log axes
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Lag time $t$ [s]", fontsize=14)
    ax.set_ylabel("MSD [$\mu$m²]", fontsize=14)
    ax.set_title(f"{condition_name} Telomere MSD", fontsize=16, fontweight='bold')

    # --- Set x-axis ticks ---
    lag_min = combined_mean.index.min()
    lag_max = combined_mean.index.max()
    lag_ticks = [50, 100, 200, 400]
    lag_ticks = [t for t in lag_ticks if lag_min <= t <= lag_max]
    ax.set_xticks(lag_ticks)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    
    ax.legend()
    ax.grid(False)
    
    return ax

# ===============================================================
# Progeria vs Control MSD comparison (Mean ± SEM)
# ===============================================================
def plot_condition_comparison_msd(msd_dict, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))

    comparison_colors = {
        'Progeria': 'slateblue',
        'Control': 'lightsteelblue'
    }

    for condition, replicate_msds in msd_dict.items():
        mean_msds = [df.mean(axis=1) for df in replicate_msds]
        combined = pd.concat(mean_msds, axis=1)
        mean = combined.mean(axis=1)
        sem = combined.sem(axis=1)

        ax.plot(mean.index, mean,
                color=comparison_colors[condition],
                linewidth=3,
                label=f"{condition} Mean ± SEM")

        ax.fill_between(mean.index,
                        mean - sem,
                        mean + sem,
                        color=comparison_colors[condition],
                        alpha=0.3)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Lag time $t$ [s]", fontsize=14)
    ax.set_ylabel("MSD [$\mu$m²]", fontsize=14)
    ax.set_title("Telomere MSD Comparison", fontsize=16, fontweight='bold')

    lag_min = min([df.index.min() for reps in msd_dict.values() for df in reps])
    lag_max = max([df.index.max() for reps in msd_dict.values() for df in reps])
    lag_ticks = [50, 100, 200, 400]
    lag_ticks = [t for t in lag_ticks if lag_min <= t <= lag_max]
    ax.set_xticks(lag_ticks)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    ax.legend()
    ax.grid(False)
    return ax

# ===============================
# FUNCTION: Combine Violin Plots
# ===============================
def plot_combined_violins(fit_files, condition_name, ax=None):
    colors = color_schemes[condition_name]
    all_params_list = []
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df['Replicate'] = f'Rep {i+1}'
        all_params_list.append(df)
    combined = pd.concat(all_params_list, ignore_index=True)
    
    combined = combined[(combined['D_eff'] > 1e-5) & 
                        (combined['alpha'] > 0.01) & 
                        (combined['sigma'] > 0.008)]
    
    combined['log_D_eff'] = np.log10(combined['D_eff'])
    
    if ax is None:
        fig, axes = plt.subplots(1,3, figsize=(15,5))
    else:
        axes = ax

    sns.violinplot(y='log_D_eff', x='Replicate', data=combined, ax=axes[0],
                   inner='box', palette=colors)
    sns.stripplot(y='log_D_eff', x='Replicate', data=combined, ax=axes[0],
                  color='k', size=3, alpha=0.25)

    logD_min = np.floor(combined['log_D_eff'].min())
    logD_max = np.ceil(combined['log_D_eff'].max())
    top_padding = 0.12 * (logD_max - logD_min)
    bottom_padding = 0.2 * (logD_max - logD_min)
    axes[0].set_ylim(logD_min - bottom_padding, logD_max + top_padding)
    axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

    axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    axes[0].set_title(f"{condition_name} Diffusion Coefficient", fontsize=16, fontweight='bold')

    axes[0].grid(False)

    sns.violinplot(y='alpha', x='Replicate', data=combined, ax=axes[1],
                   inner='box', palette=colors)
    sns.stripplot(y='alpha', x='Replicate', data=combined, ax=axes[1],
                  color='k', size=3, alpha=0.25)
    axes[1].set_ylabel(r"$\alpha$", fontsize=14)
    axes[1].set_title(f"{condition_name} Anomalous Exponent", fontsize=16, fontweight='bold')

    sns.violinplot(y='sigma', x='Replicate', data=combined, ax=axes[2],
                   inner='box', palette=colors)
    sns.stripplot(y='sigma', x='Replicate', data=combined, ax=axes[2],
                  color='k', size=3, alpha=0.25)
    axes[2].set_ylabel(r"$\sigma$ ($\mu m$)", fontsize=14)
    axes[2].set_title(f"{condition_name} Localization Error", fontsize=16, fontweight='bold')

    plt.tight_layout()
    return axes

# ===============================
# FUNCTION: Scatter D_eff vs alpha per replicate
# ===============================
def plot_D_vs_alpha_scatter(fit_files, condition_name, ax=None):
    colors = color_schemes[condition_name]
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(6,6))
    
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
        
        ax.scatter(
            df['alpha'],
            df['D_eff'],
            s=20,
            alpha=0.6,
            color=colors[i],
            label=f"Replicate {i+1}"
        )
    
    ax.set_yscale('log')
    ax.set_xlabel(r"$\alpha$", fontsize=14)
    ax.set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    ax.set_title(f"{condition_name}: $D_\\mathrm{{eff}}$ vs $\\alpha$", fontsize=16, fontweight='bold')
    
    ax.legend()
    ax.grid(False)
    plt.tight_layout()
    
    return ax

# ===============================
# FUNCTION: Scatter D_eff vs alpha for ALL replicates (both conditions)
# ===============================
def plot_all_D_vs_alpha_scatter(fit_csvs, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))
    
    for condition, files in fit_csvs.items():
        colors = color_schemes[condition]
        for i, f in enumerate(files):
            df = pd.read_csv(f)
            df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
            
            ax.scatter(
                df['alpha'],
                df['D_eff'],
                s=20,
                alpha=0.6,
                color=colors[i],
                label=f"{condition} Rep {i+1}"
            )
    
    ax.set_yscale('log')
    ax.set_xlabel(r"$\alpha$", fontsize=14)
    ax.set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    ax.set_title(r"All replicates: $D_\mathrm{eff}$ vs $\alpha$", fontsize=16, fontweight='bold')
    
    ax.legend(fontsize=9, frameon=False)
    ax.grid(False)
    plt.tight_layout()
    
    return ax

# ===============================
# LOOP THROUGH CONDITIONS
# ===============================
for cond in ['Progeria','Control']:
    msd_reps = load_and_combine_msds(replicate_csvs[cond])
    ax_msd = plot_combined_msds(msd_reps, cond)
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_{cond}_combined_MSD.svg")
    plt.close()
    
    ax_violin = plot_combined_violins(fit_csvs[cond], cond)
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_{cond}_combined_violins.svg")
    plt.close()

    ax_scatter = plot_D_vs_alpha_scatter(fit_csvs[cond], cond)
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_{cond}_D_vs_alpha_scatter.svg")
    plt.close()

# ===============================
# Progeria vs Control MSD plot
# ===============================
msd_all = {
    'Progeria': load_and_combine_msds(replicate_csvs['Progeria']),
    'Control': load_and_combine_msds(replicate_csvs['Control'])
}

ax_compare = plot_condition_comparison_msd(msd_all)
plt.tight_layout()
plt.savefig(f"{output_prefix}_Progeria_vs_Control_MSD_comparison.svg")
plt.close()

# ==============================
# FUNCTION: Load replicate medians (Not pooled)
# ==============================
def load_replicate_medians(fit_files):
    means = []
    for f in fit_files:
        df = pd.read_csv(f)
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
        means.append({
            'D_eff': df['D_eff'].median(),
            'alpha': df['alpha'].median()
        })
    return pd.DataFrame(medians)

ax_all_scatter = plot_all_D_vs_alpha_scatter(fit_csvs)
plt.tight_layout()
plt.savefig(f"{output_prefix}_ALL_D_vs_alpha_scatter.svg")
plt.close()

### Violins for drug conditions

In [ ]:
# ===============================================================
# COMBINE REPLICATES AND PLOT FINAL MSD + VIOLIN FIGURES
# ===============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker as mticker

# --- USER PARAMETERS ---
replicate_csvs = {
    'Lonafarnib': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\Lonafarnib\no backlund, 60 frames\Rep1_Lonafarnib_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\Lonafarnib\no backlund, 60 frames\Rep2_Lonafarnib_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\Lonafarnib\no backlund, 60 frames\Rep3_Lonafarnib_ALL_iMSD_values.csv'
    ],
    'UCM-13207': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\UCM-13207\no backlund, 60 frames\Rep1_UCM_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\UCM-13207\no backlund, 60 frames\Rep2_UCM_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\UCM-13207\no backlund, 60 frames\Rep3_UCM_ALL_iMSD_values.csv'
    ],
    'C75': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\C75\no backlund, 60 frames\Rep1_C75_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\C75\no backlund, 60 frames\Rep2_C75_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\C75\no backlund, 60 frames\Rep3_C75_ALL_iMSD_values.csv'
    ],
        '2 uM C75': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\2 uM C75\no backlund, 60 frames\Rep1_2uM_C75_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\2 uM C75\no backlund, 60 frames\Rep2_2uM_C75_ALL_iMSD_values.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\2 uM C75\no backlund, 60 frames\Rep3_2uM_C75_ALL_iMSD_values.csv'
    ]
}

fit_csvs = {
    'Lonafarnib': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\Lonafarnib\no backlund, 60 frames\Rep1_Lonafarnib_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\Lonafarnib\no backlund, 60 frames\Rep2_Lonafarnib_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\Lonafarnib\no backlund, 60 frames\Rep3_Lonafarnib_ALL_fit_params.csv'
    ],
    'UCM-13207': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\UCM-13207\no backlund, 60 frames\Rep1_UCM_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\UCM-13207\no backlund, 60 frames\Rep2_UCM_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\UCM-13207\no backlund, 60 frames\Rep3_UCM_ALL_fit_params.csv'
    ],
    'C75': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\C75\no backlund, 60 frames\Rep1_C75_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\C75\no backlund, 60 frames\Rep2_C75_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\C75\no backlund, 60 frames\Rep3_C75_ALL_fit_params.csv'
    ],
        '2 uM C75': [
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\2 uM C75\no backlund, 60 frames\Rep1_2uM_C75_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\2 uM C75\no backlund, 60 frames\Rep2_2uM_C75_ALL_fit_params.csv',
        r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\2 uM C75\no backlund, 60 frames\Rep3_2uM_C75_ALL_fit_params.csv'
    ]
}

output_prefix = r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\drugs\output\no backlund, 60 frames\final_combined_figures'

# ==============================
# COLOR SCHEMES PER CONDITION
# ==============================
color_schemes = {
    'Lonafarnib': ["cadetblue", "teal", "seagreen"],
    'UCM-13207': ["mediumaquamarine", "mediumseagreen", "lightseagreen"],
    'C75': ["forestgreen", "darkgreen", "green"],
    '2 uM C75': ["cadetblue", "lightblue", "steelblue"]
}

# ==============================
# FUNCTION: Combine MSD replicates
# ==============================
def load_and_combine_msds(msd_files):
    replicate_msds = []
    for f in msd_files:
        df = pd.read_csv(f, index_col=0)
        replicate_msds.append(df)
    return replicate_msds

def plot_combined_msds(replicate_msds, condition_name, ax=None):
    colors = color_schemes[condition_name]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))
    
    mean_msds = []
    for df in replicate_msds:
        mean_msds.append(df.mean(axis=1))
        ax.plot(df.index, df, color='gray', alpha=0.05)
    
    for i, median_df in enumerate(mean_msds):
        ax.plot(median_df.index, median_df, color=colors[i], linewidth=2, label=f'Replicate {i+1}')
    
    combined = pd.concat(mean_msds, axis=1)
    combined_mean = combined.mean(axis=1)
    combined_sem = combined.sem(axis=1)
    
    ax.plot(combined_mean.index, combined_mean, color='k', linewidth=3, label='Mean ± SEM')
    ax.fill_between(combined_mean.index,
                    combined_mean - combined_sem,
                    combined_mean + combined_sem,
                    color='k', alpha=0.35)
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Lag time $t$ [s]", fontsize=14)
    ax.set_ylabel("MSD [$\mu$m²]", fontsize=14)
    ax.set_title(f"{condition_name} Telomere MSD", fontsize=16, fontweight='bold')

    lag_min = combined_mean.index.min()
    lag_max = combined_mean.index.max()
    lag_ticks = [50, 100, 200, 400]
    lag_ticks = [t for t in lag_ticks if lag_min <= t <= lag_max]
    ax.set_xticks(lag_ticks)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    
    ax.legend()
    ax.grid(False)
    return ax

# ===============================================================
# Lonafarnib vs UCM-13207 vs C75 vs 2 uM C75 MSD comparison (Mean ± SEM)
# ===============================================================
def plot_condition_comparison_msd(msd_dict, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))

    comparison_colors = {
        'Lonafarnib': 'teal',
        'UCM-13207': 'mediumaquamarine',
        'C75': 'forestgreen',
        '2 uM C75': 'midnightblue'
    }

    for condition, replicate_msds in msd_dict.items():
        mean_msds = [df.mean(axis=1) for df in replicate_msds]
        combined = pd.concat(mean_msds, axis=1)
        mean = combined.mean(axis=1)
        sem = combined.sem(axis=1)

        ax.plot(mean.index, mean,
                color=comparison_colors[condition],
                linewidth=3,
                label=f"{condition} Mean ± SEM")

        ax.fill_between(mean.index,
                        mean - sem,
                        mean + sem,
                        color=comparison_colors[condition],
                        alpha=0.3)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Lag time $t$ [s]", fontsize=14)
    ax.set_ylabel("MSD [$\mu$m²]", fontsize=14)
    ax.set_title("Telomere MSD Comparison", fontsize=16, fontweight='bold')

    lag_min = min([df.index.min() for reps in msd_dict.values() for df in reps])
    lag_max = max([df.index.max() for reps in msd_dict.values() for df in reps])
    lag_ticks = [50, 100, 200, 400]
    lag_ticks = [t for t in lag_ticks if lag_min <= t <= lag_max]
    ax.set_xticks(lag_ticks)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    ax.legend()
    ax.grid(False)
    return ax

# ===============================
# FUNCTION: Combine Violin Plots
# ===============================
def plot_combined_violins(fit_files, condition_name, ax=None):
    colors = color_schemes[condition_name]
    all_params_list = []
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df['Replicate'] = f'Rep {i+1}'
        all_params_list.append(df)
    combined = pd.concat(all_params_list, ignore_index=True)
    
    combined = combined[(combined['D_eff'] > 1e-5) & 
                        (combined['alpha'] > 0.01) & 
                        (combined['sigma'] > 0.008)]
    
    combined['log_D_eff'] = np.log10(combined['D_eff'])
    
    if ax is None:
        fig, axes = plt.subplots(1,3, figsize=(15,5))
    else:
        axes = ax

    sns.violinplot(y='log_D_eff', x='Replicate', data=combined, ax=axes[0],
                   inner='box', palette=colors)
    sns.stripplot(y='log_D_eff', x='Replicate', data=combined, ax=axes[0],
                  color='k', size=3, alpha=0.25)

    logD_min = np.floor(combined['log_D_eff'].min())
    logD_max = np.ceil(combined['log_D_eff'].max())
    top_padding = 0.12 * (logD_max - logD_min)
    bottom_padding = 0.2 * (logD_max - logD_min)
    axes[0].set_ylim(logD_min - bottom_padding, logD_max + top_padding)
    axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

    axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    axes[0].set_title(f"{condition_name} Diffusion Coefficient", fontsize=16, fontweight='bold')

    axes[0].grid(False)

    sns.violinplot(y='alpha', x='Replicate', data=combined, ax=axes[1],
                   inner='box', palette=colors)
    sns.stripplot(y='alpha', x='Replicate', data=combined, ax=axes[1],
                  color='k', size=3, alpha=0.25)
    axes[1].set_ylabel(r"$\alpha$", fontsize=14)
    axes[1].set_title(f"{condition_name} Anomalous Exponent", fontsize=16, fontweight='bold')

    sns.violinplot(y='sigma', x='Replicate', data=combined, ax=axes[2],
                   inner='box', palette=colors)
    sns.stripplot(y='sigma', x='Replicate', data=combined, ax=axes[2],
                  color='k', size=3, alpha=0.25)
    axes[2].set_ylabel(r"$\sigma$ ($\mu m$)", fontsize=14)
    axes[2].set_title(f"{condition_name} Localization Error", fontsize=16, fontweight='bold')

    plt.tight_layout()
    return axes

# ===============================
# LOOP THROUGH CONDITIONS
# ===============================
for cond in ['Lonafarnib','UCM-13207','C75', '2 uM C75']:
    msd_reps = load_and_combine_msds(replicate_csvs[cond])
    ax_msd = plot_combined_msds(msd_reps, cond)
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_{cond}_combined_MSD.svg")
    plt.close()
    
    ax_violin = plot_combined_violins(fit_csvs[cond], cond)
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_{cond}_combined_violins.svg")
    plt.close()

# ===============================
# MSD comparison across drugs
# ===============================
msd_all = {
    'Lonafarnib': load_and_combine_msds(replicate_csvs['Lonafarnib']),
    'UCM-13207': load_and_combine_msds(replicate_csvs['UCM-13207']),
    'C75': load_and_combine_msds(replicate_csvs['C75']),
    '2 uM C75': load_and_combine_msds(replicate_csvs['2 uM C75'])
}

ax_compare = plot_condition_comparison_msd(msd_all)
plt.tight_layout()
plt.savefig(f"{output_prefix}_Lonafarnib_vs_UCM13207_vs_C75_vs_2uMC75_MSD_comparison.svg")
plt.close()

# ===============================================================
# FINAL: MSD comparison across ALL 5 conditions
# ===============================================================
def plot_all_conditions_msd(msd_dict, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))

    comparison_colors = {
        'Progeria': 'slateblue',
        'Control': 'lightsteelblue',
        'Lonafarnib': 'teal',
        'UCM-13207': 'mediumaquamarine',
        'C75': 'forestgreen',
        '2 uM C75': 'midnightblue'
    }

    for condition, replicate_msds in msd_dict.items():
        mean_msds = [df.mean(axis=1) for df in replicate_msds]
        combined = pd.concat(mean_msds, axis=1)
        mean = combined.mean(axis=1)
        sem = combined.sem(axis=1)

        ax.plot(mean.index, mean,
                color=comparison_colors.get(condition, 'k'),
                linewidth=3,
                label=f"{condition} Mean ± SEM")

        ax.fill_between(mean.index,
                        mean - sem,
                        mean + sem,
                        color=comparison_colors.get(condition, 'k'),
                        alpha=0.25)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Lag time $t$ [s]", fontsize=14)
    ax.set_ylabel("MSD [$\mu$m²]", fontsize=14)
    ax.set_title("Telomere MSD Comparison (All Conditions)", fontsize=16, fontweight='bold')

    lag_min = min([df.index.min() for reps in msd_dict.values() for df in reps])
    lag_max = max([df.index.max() for reps in msd_dict.values() for df in reps])
    lag_ticks = [50, 100, 200, 400]
    lag_ticks = [t for t in lag_ticks if lag_min <= t <= lag_max]
    ax.set_xticks(lag_ticks)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    ax.legend()
    ax.grid(False)

    return ax


# ===============================
# LOAD ALL MSDs TOGETHER
# ===============================
msd_all_conditions = {
    'Progeria': load_and_combine_msds(
        [
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep1_HGPS_ALL_iMSD_values.csv',
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep2_HGPS_ALL_iMSD_values.csv',
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\HGPS\no backlund\60 frames\Rep3_HGPS_ALL_iMSD_values.csv'
        ]
    ),
    'Control': load_and_combine_msds(
        [
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep1_Controls_ALL_iMSD_values.csv',
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep2_Controls_ALL_iMSD_values.csv',
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep3_Controls_ALL_iMSD_values.csv'
        ]
    ),
    'Lonafarnib': load_and_combine_msds(replicate_csvs['Lonafarnib']),
    'UCM-13207': load_and_combine_msds(replicate_csvs['UCM-13207']),
    'C75': load_and_combine_msds(replicate_csvs['C75']),
    '2 uM C75': load_and_combine_msds(replicate_csvs['2 uM C75'])
}

# ===============================
# FINAL PLOT: ALL CONDITIONS
# ===============================
ax_all = plot_all_conditions_msd(msd_all_conditions)
plt.tight_layout()
plt.savefig(r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Output\no backlund\60 frames\Telomere_MSD_ALL_6_conditions.svg')
plt.close()

from matplotlib.ticker import FuncFormatter

# ===============================================================
# ZOOMED MSD COMPARISON: Control vs C75 vs 2 uM C75
# ===============================================================
def plot_control_c75_zoom(msd_dict, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,7))

    zoom_colors = {
        'Control': 'lightsteelblue',
        'C75': 'forestgreen',
        '2 uM C75': 'midnightblue'
    }

    for condition, replicate_msds in msd_dict.items():
        mean_msds = [df.mean(axis=1) for df in replicate_msds]
        combined = pd.concat(mean_msds, axis=1)
        mean = combined.mean(axis=1)
        sem = combined.sem(axis=1)

        ax.plot(mean.index, mean,
                color=zoom_colors[condition],
                linewidth=3,
                label=f"{condition} Mean ± SEM")

        ax.fill_between(mean.index,
                        mean - sem,
                        mean + sem,
                        color=zoom_colors[condition],
                        alpha=0.3)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel("Lag time $t$ [s]", fontsize=14)
    ax.set_ylabel("MSD [$\mu$m²]", fontsize=14)
    ax.set_title("Telomere MSD (Zoom: Control vs C75)", fontsize=16, fontweight='bold')

    # ---- ZOOM LIMITS ----
    ax.set_xlim(right=150)
    ax.set_ylim(top=3e-2)

    # ---- X-TICK LABELS ----
    lag_ticks = [50, 100]
    ax.set_xticks(lag_ticks)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x)}"))

    ax.legend()
    ax.grid(False)

    return ax


msd_zoom = {
    'Control': load_and_combine_msds(
        [
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep1_Controls_ALL_iMSD_values.csv',
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep2_Controls_ALL_iMSD_values.csv',
            r'D:\Research Projects\Progeria\Chromatin Tracking\Data\20251210 Plotting Replicates\Controls\no backlund\60 frames\Rep3_Controls_ALL_iMSD_values.csv'
        ]
    ),
    'C75': load_and_combine_msds(replicate_csvs['C75']),
    '2 uM C75': load_and_combine_msds(replicate_csvs['2 uM C75'])
}

ax_zoom = plot_control_c75_zoom(msd_zoom)
plt.tight_layout()
plt.savefig(f"{output_prefix}_Control_vs_C75_vs_2uMC75_MSD_zoom.svg")
plt.close()


### Batch Effect Check (at telomere track level)

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects"

fit_csvs = {
    "UCM": [
        rf"{base}\Rep1_UCM_ALL_fit_params.csv",
        rf"{base}\Rep2_UCM_ALL_fit_params.csv",
        rf"{base}\Rep3_UCM_ALL_fit_params.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep2_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep3_HGPS_ALL_fit_params.csv",
    ]
}

scan_csvs = {
    "UCM": [
        rf"{base}\Rep1_UCM_scan_areas.csv",
        rf"{base}\Rep2_UCM_scan_areas.csv",
        rf"{base}\Rep3_UCM_scan_areas.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_scan_areas.csv",
        rf"{base}\Rep2_HGPS_scan_areas.csv",
        rf"{base}\Rep3_HGPS_scan_areas.csv",
    ]
}

# -----------------------------
# Batch effect function
# -----------------------------
def run_batch_variance_test(fit_files, scan_files, condition_name=""):
    print("\n" + "="*60)
    print(f"Batch variance test for {condition_name}")
    print("="*60)

    # ---- Fit parameters ----
    dfs = []
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
        df['Replicate'] = f"Rep{i+1}"
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)


    for col in ['D_eff', 'alpha']:
    
        model = smf.mixedlm(f"{col} ~ 1", combined, groups=combined["Replicate"]).fit()
        rep_var = model.cov_re.iloc[0, 0]
        resid_var = model.scale
        frac = rep_var / (rep_var + resid_var)
        print(f"{col}: replicate variance fraction = {frac:.3f}")

    # ---- Scan area ----
    scan_dfs = []
    for i, f in enumerate(scan_files):
        df = pd.read_csv(f)
        df['Replicate'] = f"Rep{i+1}"
        scan_dfs.append(df)

    scan_combined = pd.concat(scan_dfs, ignore_index=True)

    model = smf.mixedlm("scan_area_um2 ~ 1", scan_combined, groups=scan_combined["Replicate"]).fit()
    rep_var = model.cov_re.iloc[0, 0]
    resid_var = model.scale
    frac = rep_var / (rep_var + resid_var)
    print(f"scan_area_um2: replicate variance fraction = {frac:.3f}")

# -----------------------------
# Run for each condition
# -----------------------------
for cond in ["UCM", "HGPS"]:
    run_batch_variance_test(fit_csvs[cond], scan_csvs[cond], cond)
    


### Batch effect check (at cell level (median per cell))

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects"

fit_csvs = {
    "C75": [
        rf"{base}\Rep1_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_C75_ALL_fit_params.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep2_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep3_HGPS_ALL_fit_params.csv",
    ]
}

scan_csvs = {
    "C75": [
        rf"{base}\Rep1_C75_scan_areas.csv",
        rf"{base}\Rep2_C75_scan_areas.csv",
        rf"{base}\Rep3_C75_scan_areas.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_scan_areas.csv",
        rf"{base}\Rep2_HGPS_scan_areas.csv",
        rf"{base}\Rep3_HGPS_scan_areas.csv",
    ]
}

# -----------------------------
# Batch effect function
# -----------------------------
def run_batch_variance_test(fit_files, scan_files, condition_name=""):
    print("\n" + "="*60)
    print(f"Batch variance test for {condition_name} (cell-median)")
    print("="*60)

    # ---- Fit parameters ----
    dfs = []
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
        df['Replicate'] = f"Rep{i+1}"

        # Average tracks per cell
        df_cell = (
            df.groupby(['Replicate', 'cell_id'], as_index=False)
              .agg({'D_eff': 'median', 'alpha': 'median'})
        )

        dfs.append(df_cell)

    combined = pd.concat(dfs, ignore_index=True)

    for col in ['D_eff', 'alpha']:
        model = smf.mixedlm(f"{col} ~ 1", combined, groups=combined["Replicate"]).fit()
        rep_var = model.cov_re.iloc[0, 0]
        resid_var = model.scale
        frac = rep_var / (rep_var + resid_var)
        print(f"{col}: replicate variance fraction = {frac:.3f}")

    # ---- Scan area ----
    scan_dfs = []
    for i, f in enumerate(scan_files):
        df = pd.read_csv(f)
        df['Replicate'] = f"Rep{i+1}"

        # Average scan area per cell
        df_cell = (
            df.groupby(['Replicate', 'cell_id'], as_index=False)
              .agg({'scan_area_um2': 'median'})
        )

        scan_dfs.append(df_cell)

    scan_combined = pd.concat(scan_dfs, ignore_index=True)

    try:
        model = smf.mixedlm("scan_area_um2 ~ 1", scan_combined, groups=scan_combined["Replicate"]).fit()
        rep_var = model.cov_re.iloc[0, 0]
        resid_var = model.scale
        frac = rep_var / (rep_var + resid_var)
    except Exception as e:
        print("scan_area_um2: replicate variance not identifiable (≈ 0)")
        frac = 0.0

    print(f"scan_area_um2: replicate variance fraction = {frac:.3f}")


# -----------------------------
# Run for each condition
# -----------------------------
for cond in ["C75", "HGPS"]:
    run_batch_variance_test(fit_csvs[cond], scan_csvs[cond], cond)


### Batch effect using one way ANOVA (all telomere tracks level)

In [ ]:
import pandas as pd
from scipy.stats import f_oneway

# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects"

fit_csvs = {
    "C75": [
        rf"{base}\Rep1_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_C75_ALL_fit_params.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep2_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep3_HGPS_ALL_fit_params.csv",
    ]
}

scan_csvs = {
    "C75": [
        rf"{base}\Rep1_C75_scan_areas.csv",
        rf"{base}\Rep2_C75_scan_areas.csv",
        rf"{base}\Rep3_C75_scan_areas.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_scan_areas.csv",
        rf"{base}\Rep2_HGPS_scan_areas.csv",
        rf"{base}\Rep3_HGPS_scan_areas.csv",
    ]
}

# -----------------------------
# One-way ANOVA function (track-level, not per-cell)
# -----------------------------
def run_batch_anova(fit_files, scan_files, condition_name=""):
    print("\n" + "="*60)
    print(f"Batch ANOVA for {condition_name} (all tracks)")
    print("="*60)

    # ---- Fit parameters ----
    dfs = []
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
        df['Replicate'] = f"Rep{i+1}"

        # No grouping, keep all tracks
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)

    for col in ['D_eff', 'alpha']:
        # Prepare data for ANOVA
        groups = [combined.loc[combined['Replicate'] == f"Rep{i+1}", col] for i in range(len(fit_files))]
        f_stat, p_val = f_oneway(*groups)
        print(f"{col}: F = {f_stat:.3f}, p = {p_val:.3e}")

    # ---- Scan area ----
    scan_dfs = []
    for i, f in enumerate(scan_files):
        df = pd.read_csv(f)
        df['Replicate'] = f"Rep{i+1}"

        # Keep all scan areas
        scan_dfs.append(df)

    scan_combined = pd.concat(scan_dfs, ignore_index=True)

    groups = [scan_combined.loc[scan_combined['Replicate'] == f"Rep{i+1}", 'scan_area_um2'] 
              for i in range(len(scan_files))]
    f_stat, p_val = f_oneway(*groups)
    print(f"scan_area_um2: F = {f_stat:.3f}, p = {p_val:.3e}")
    
# -----------------------------
# Run ANOVA for each condition
# -----------------------------
for cond in ["C75", "HGPS"]:
    run_batch_anova(fit_csvs[cond], scan_csvs[cond], cond)
    

### Batch effect using one way ANOVA (cell median level)

In [ ]:
import pandas as pd
from scipy.stats import f_oneway

# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects\no backlund\60 frames"

fit_csvs = {
    "Lonafarnib": [
        rf"{base}\Rep1_Lonafarnib_ALL_fit_params.csv",
        rf"{base}\Rep2_Lonafarnib_ALL_fit_params.csv",
        rf"{base}\Rep3_Lonafarnib_ALL_fit_params.csv",
    ],
    "C75": [
        rf"{base}\Rep1_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_C75_ALL_fit_params.csv",
    ]
}

scan_csvs = {
    "Lonafarnib": [
        rf"{base}\Rep1_Lonafarnib_scan_areas.csv",
        rf"{base}\Rep2_Lonafarnib_scan_areas.csv",
        rf"{base}\Rep3_Lonafarnib_scan_areas.csv",
    ],
    "C75": [
        rf"{base}\Rep1_C75_scan_areas.csv",
        rf"{base}\Rep2_C75_scan_areas.csv",
        rf"{base}\Rep3_C75_scan_areas.csv",
    ]
}

# -----------------------------
# One-way ANOVA function
# -----------------------------
def run_batch_anova(fit_files, scan_files, condition_name=""):
    print("\n" + "="*60)
    print(f"Batch ANOVA for {condition_name} (cell-median)")
    print("="*60)

    # ---- Fit parameters ----
    dfs = []
    for i, f in enumerate(fit_files):
        df = pd.read_csv(f)
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]
        df['Replicate'] = f"Rep{i+1}"

        # Average tracks per cell
        df_cell = (
            df.groupby(['Replicate', 'cell_id'], as_index=False)
              .agg({'D_eff': 'median', 'alpha': 'median'})
        )
        dfs.append(df_cell)

    combined = pd.concat(dfs, ignore_index=True)

    for col in ['D_eff', 'alpha']:
        # Prepare data for ANOVA
        groups = [combined.loc[combined['Replicate'] == f"Rep{i+1}", col] for i in range(len(fit_files))]
        f_stat, p_val = f_oneway(*groups)
        print(f"{col}: F = {f_stat:.3f}, p = {p_val:.3e}")

    # ---- Scan area ----
    scan_dfs = []
    for i, f in enumerate(scan_files):
        df = pd.read_csv(f)
        df['Replicate'] = f"Rep{i+1}"

        # average scan area per cell
        df_cell = (
            df.groupby(['Replicate', 'cell_id'], as_index=False)
              .agg({'scan_area_um2': 'median'})
        )
        scan_dfs.append(df_cell)

    scan_combined = pd.concat(scan_dfs, ignore_index=True)

    groups = [scan_combined.loc[scan_combined['Replicate'] == f"Rep{i+1}", 'scan_area_um2'] 
              for i in range(len(scan_files))]
    f_stat, p_val = f_oneway(*groups)
    print(f"scan_area_um2: F = {f_stat:.3f}, p = {p_val:.3e}")


# -----------------------------
# Run ANOVA for each condition
# -----------------------------
for cond in ["Lonafarnib", "C75"]:
    run_batch_anova(fit_csvs[cond], scan_csvs[cond], cond)


### Significance test with a linear mixed effects model

In [ ]:
import os
import pandas as pd
import statsmodels.formula.api as smf

# ==========================================================
# Paths
# ==========================================================
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects\no backlund\60 frames"

fit_csvs = {
    "Controls": [
        rf"{base}\Rep1_Controls_ALL_fit_params.csv",
        rf"{base}\Rep2_Controls_ALL_fit_params.csv",
        rf"{base}\Rep3_Controls_ALL_fit_params.csv",
    ],
    "C75": [
        rf"{base}\Rep1_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_C75_ALL_fit_params.csv",
    ]
}

scan_csvs = {
    "Controls": [
        rf"{base}\Rep1_Controls_scan_areas.csv",
        rf"{base}\Rep2_Controls_scan_areas.csv",
        rf"{base}\Rep3_Controls_scan_areas.csv",
    ],
    "C75": [
        rf"{base}\Rep1_C75_scan_areas.csv",
        rf"{base}\Rep2_C75_scan_areas.csv",
        rf"{base}\Rep3_C75_scan_areas.csv",
    ]
}

all_cells = []

# -----------------------------
# Fit parameters (D_eff, alpha)
# -----------------------------
for condition, files in fit_csvs.items():
    for i, f in enumerate(files):
        df = pd.read_csv(f)

        # Same filtering as before
        df = df[(df['D_eff'] > 1e-5) & (df['alpha'] > 0.01)]

        df['replicate'] = f"Rep{i+1}"
        df['condition'] = condition

        # Cell-level medians
        df_cell = (
            df.groupby(['replicate', 'condition', 'cell_id'], as_index=False)
              .agg({
                  'D_eff': 'median',
                  'alpha': 'median'
              })
        )

        all_cells.append(df_cell)

fit_combined = pd.concat(all_cells, ignore_index=True)

scan_cells = []

for condition, files in scan_csvs.items():
    for i, f in enumerate(files):
        df = pd.read_csv(f)
        df['replicate'] = f"Rep{i+1}"
        df['condition'] = condition

        df_cell = (
            df.groupby(['replicate', 'condition', 'cell_id'], as_index=False)
              .agg({'scan_area_um2': 'median'})
        )

        scan_cells.append(df_cell)

scan_combined = pd.concat(scan_cells, ignore_index=True)

model_D = smf.mixedlm(
    "D_eff ~ condition",
    data=fit_combined,
    groups=fit_combined["replicate"]
)
result_D = model_D.fit()

print("\n" + "="*60)
print("LMM: D_eff ~ condition + (1|replicate)")
print("="*60)
print(result_D.summary())

model_alpha = smf.mixedlm(
    "alpha ~ condition",
    data=fit_combined,
    groups=fit_combined["replicate"]
)
result_alpha = model_alpha.fit()

print("\n" + "="*60)
print("LMM: alpha ~ condition + (1|replicate)")
print("="*60)
print(result_alpha.summary())

model_scan = smf.mixedlm(
    "scan_area_um2 ~ condition",
    data=scan_combined,
    groups=scan_combined["replicate"]
)
result_scan = model_scan.fit()

print("\n" + "="*60)
print("LMM: scan_area_um2 ~ condition + (1|replicate)")
print("="*60)
print(result_scan.summary())

def print_lmm_full(result):
    print("-"*70)
    print(f"{'Parameter':<30}{'Coef':>12}{'Std.Err':>12}{'z':>12}{'P>|z|':>15}")
    print("-"*70)

    for name, coef, se, zval, pval in zip(
            result.params.index,
            result.params.values,
            result.bse.values,
            result.tvalues.values,
            result.pvalues.values):
        print(f"{name:<30}{coef:12.6f}{se:12.6f}{zval:12.6f}{pval:15.3e}")

print_lmm_full(result_D)
print_lmm_full(result_alpha)
print_lmm_full(result_scan)


### POOLED VIOLINS + MIXED EFFECTS TEST (Controls vs HGPS)

In [ ]:
#### At the telomere track level

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker as mticker
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu
from matplotlib.ticker import MultipleLocator


# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects"

fit_csvs = {
    "Controls": [
        rf"{base}\Rep1_Controls_ALL_fit_params.csv",
        rf"{base}\Rep2_Controls_ALL_fit_params.csv",
        rf"{base}\Rep3_Controls_ALL_fit_params.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep2_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep3_HGPS_ALL_fit_params.csv",
    ],
    "Lonafarnib": [
        rf"{base}\Rep1_Lonafarnib_ALL_fit_params.csv",
        rf"{base}\Rep2_Lonafarnib_ALL_fit_params.csv",
        rf"{base}\Rep3_Lonafarnib_ALL_fit_params.csv",
    ],
    "UCM-13207": [
        rf"{base}\Rep1_UCM_ALL_fit_params.csv",
        rf"{base}\Rep2_UCM_ALL_fit_params.csv",
        rf"{base}\Rep3_UCM_ALL_fit_params.csv",
    ],
    "C75": [
        rf"{base}\Rep1_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_C75_ALL_fit_params.csv",
    ]
}

# -----------------------------
# Pool fit CSVs into one DataFrame
# -----------------------------
pooled_all = []

for condition, files in fit_csvs.items():
    for i, fpath in enumerate(files, 1):
        df = pd.read_csv(fpath)
        df['Condition'] = condition
        df['Replicate'] = f"Rep{i}"
        pooled_all.append(df)

pooled_all = pd.concat(pooled_all, ignore_index=True)

pooled_all = pooled_all[
    (pooled_all['D_eff'] > 1e-5) &
    (pooled_all['alpha'] > 0.01) &
    (pooled_all['sigma'] > 0.001)
]

# -----------------------------
# Function: plot violins + stats below x-axis
# -----------------------------
def plot_pooled_violins_with_stats(pooled_df, output_path):
    fig, axes = plt.subplots(1, 2, figsize=(12,6))

    # Add log column for D_eff
    pooled_df = pooled_df.copy()
    pooled_df['log_D_eff'] = np.log10(pooled_df['D_eff'])

    # -----------------------------
    # Run mixed-effects models
    # -----------------------------
    stats_dict = {}
    for col in ['D_eff', 'alpha']:
        model = smf.mixedlm(f"{col} ~ Condition", pooled_df, groups=pooled_df["Replicate"]).fit()
        coef = model.params['Condition[T.HGPS]']
        ci_lower, ci_upper = model.conf_int().loc['Condition[T.HGPS]']
        stats_dict[col] = {'coef': coef, 'ci': (ci_lower, ci_upper)}

    # -----------------------------
    # Run Mann–Whitney U on pooled tracks
    # -----------------------------
    mannwhitney_dict = {}
    for col in ['D_eff', 'alpha']:
        cond1 = pooled_df[pooled_df['Condition'] == 'Controls'][col]
        cond2 = pooled_df[pooled_df['Condition'] == 'HGPS'][col]
        stat, pval = mannwhitneyu(cond1, cond2, alternative='two-sided')
        mannwhitney_dict[col] = {'stat': stat, 'pval': pval}

    # -----------------------------
    # Count number of cells, tracks, and mean per condition
    # -----------------------------
    counts = {}
    for cond in ['Controls', 'HGPS']:
        subset = pooled_df[pooled_df['Condition'] == cond]
        n_cells = subset['cell_id'].nunique() if 'cell_id' in subset.columns else 'N/A'
        n_tracks = len(subset)
        median_D = subset['D_eff'].median()
        median_alpha = subset['alpha'].median()
        counts[cond] = {'cells': n_cells, 'tracks': n_tracks, 'median_D': median_D, 'median_alpha': median_alpha}

    # -----------------------------
    # D_eff violin
    # -----------------------------
    sns.violinplot(
        y='log_D_eff', x='Condition', data=pooled_df, ax=axes[0],
        inner='box', palette=["lightsteelblue", "mediumorchid"]
    )
    sns.stripplot(
        y='log_D_eff', x='Condition', data=pooled_df, ax=axes[0],
        color='k', size=3, alpha=0.25
    )
    axes[0].margins(y=0.02)
    axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    axes[0].set_title("Diffusion Coefficient", fontsize=16, fontweight='bold')
    axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))
    axes[0].grid(False)

    # Add stats + counts + means below x-axis
    coef = stats_dict['D_eff']['coef']
    ci_lower, ci_upper = stats_dict['D_eff']['ci']
    pval = mannwhitney_dict['D_eff']['pval']
    axes[0].text(0.5, -0.3,
                 f"Mixed-effect coef: {coef:.3g}, 95% CI: [{ci_lower:.3g}, {ci_upper:.3g}]\n"
                 f"Mann–Whitney p = {pval:.3g}\n"
                 f"Controls: {counts['Controls']['cells']} cells, {counts['Controls']['tracks']} tracks, mean D = {counts['Controls']['median_D']:.3g}\n"
                 f"HGPS: {counts['HGPS']['cells']} cells, {counts['HGPS']['tracks']} tracks, mean D = {counts['HGPS']['median_D']:.3g}",
                 ha='center', va='top', fontsize=10, transform=axes[0].transAxes)

    # -----------------------------
    # alpha violin
    # -----------------------------
    sns.violinplot(
        y='alpha', x='Condition', data=pooled_df, ax=axes[1],
        inner='box', palette=["lightsteelblue", "mediumorchid"]
    )
    sns.stripplot(
        y='alpha', x='Condition', data=pooled_df, ax=axes[1],
        color='k', size=3, alpha=0.25
    )
    axes[1].margins(y=0.02)
    axes[1].set_ylabel(r"$\alpha$", fontsize=14)
    axes[1].set_title("Anomalous Exponent", fontsize=16, fontweight='bold')
    axes[1].grid(False)

    # Add stats + counts + means below x-axis
    coef = stats_dict['alpha']['coef']
    ci_lower, ci_upper = stats_dict['alpha']['ci']
    pval = mannwhitney_dict['alpha']['pval']
    axes[1].text(0.5, -0.3,
                 f"Mixed-effect coef: {coef:.3g}, 95% CI: [{ci_lower:.3g}, {ci_upper:.3g}]\n"
                 f"Mann–Whitney p = {pval:.3g}\n"
                 f"Controls: {counts['Controls']['cells']} cells, {counts['Controls']['tracks']} tracks, mean α = {counts['Controls']['median_alpha']:.3g}\n"
                 f"HGPS: {counts['HGPS']['cells']} cells, {counts['HGPS']['tracks']} tracks, mean α = {counts['HGPS']['median_alpha']:.3g}",
                 ha='center', va='top', fontsize=10, transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()
    plt.close(fig)

# -----------------------------
# Run original figure
# -----------------------------
output_file = rf"{base}\Pooled_Controls_vs_HGPS_violins_with_stats.svg"
plot_pooled_violins_with_stats(
    pooled_all[pooled_all["Condition"].isin(["Controls", "HGPS"])],
    output_file
)

# -----------------------------
# Run second figure: drugs only (same stats logic disabled because function assumes Controls vs HGPS)
# -----------------------------
drug_df = pooled_all[pooled_all["Condition"].isin(["Lonafarnib", "UCM-13207", "C75"])]

sns.set(style="white")

fig, axes = plt.subplots(1, 2, figsize=(12,6))
drug_df = drug_df.copy()
drug_df["log_D_eff"] = np.log10(drug_df["D_eff"])

drug_palette = ["teal", "mediumaquamarine", "mediumseagreen"]

sns.violinplot(y="log_D_eff", x="Condition", data=drug_df, ax=axes[0],
               inner="box", palette=drug_palette)
sns.stripplot(y="log_D_eff", x="Condition", data=drug_df, ax=axes[0],
              color="k", size=3, alpha=0.25)

axes[0].margins(y=0.02)
axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)")
axes[0].set_title("Diffusion Coefficient (Drugs)", fontsize=16, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))


sns.violinplot(y="alpha", x="Condition", data=drug_df, ax=axes[1],
               inner="box", palette=drug_palette)
sns.stripplot(y="alpha", x="Condition", data=drug_df, ax=axes[1],
              color="k", size=3, alpha=0.25)

axes[1].margins(y=0.02)

axes[1].set_ylabel(r"$\alpha$")
axes[1].set_title("Anomalous Exponent (Drugs)", fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig(rf"{base}\Pooled_Drugs_only_violins.svg")
plt.show()
plt.close(fig)

# -----------------------------
# Run third figure: all conditions together
# -----------------------------
all_df = pooled_all[
    pooled_all["Condition"].isin(["Controls", "HGPS", "Lonafarnib", "UCM-13207", "C75"])
].copy()

all_df["log_D_eff"] = np.log10(all_df["D_eff"])

all_palette = [
    "lightsteelblue",   # Controls
    "slateblue",        # HGPS
    "teal",             # Lonafarnib
    "mediumseagreen",   # UCM-13207
    "mediumaquamarine"  # C75
]

condition_order = ["Controls", "HGPS", "Lonafarnib", "UCM-13207", "C75"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(
    y="log_D_eff", x="Condition", data=all_df, ax=axes[0],
    inner="box", palette=all_palette, order=condition_order
)
sns.stripplot(
    y="log_D_eff", x="Condition", data=all_df, ax=axes[0],
    color="k", size=3, alpha=0.25, order=condition_order
)

axes[0].margins(y=0.02)
axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)")
axes[0].set_title("Diffusion Coefficient (All Conditions)", fontsize=16, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

sns.violinplot(
    y="alpha", x="Condition", data=all_df, ax=axes[1],
    inner="box", palette=all_palette, order=condition_order
)
sns.stripplot(
    y="alpha", x="Condition", data=all_df, ax=axes[1],
    color="k", size=3, alpha=0.25, order=condition_order
)

axes[1].margins(y=0.02)
axes[1].set_ylabel(r"$\alpha$")
axes[1].set_title("Anomalous Exponent (All Conditions)", fontsize=16, fontweight='bold')


plt.tight_layout()
plt.savefig(rf"{base}\Pooled_All_Conditions_violins.svg")
plt.show()
plt.close(fig)



In [ ]:
#### At the per-cell median level

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker as mticker
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu
from matplotlib.ticker import MultipleLocator

# -----------------------------
# Mann–Whitney vs HGPS for drugs
# -----------------------------
def add_mannwhitney_vs_hgps(ax, pooled_df, value_col,
                           y_text_start=-0.30, dy=0.12):

    baseline = pooled_df[pooled_df["Condition"] == "HGPS"][value_col]
    drugs = ["Lonafarnib", "UCM-13207", "C75", "2 uM C75"]

    y = y_text_start
    for drug in drugs:
        drug_vals = pooled_df[pooled_df["Condition"] == drug][value_col]
        if len(drug_vals) == 0:
            continue

        stat, pval = mannwhitneyu(baseline, drug_vals, alternative="two-sided")

        ax.text(
            0.5, y,
            f"{drug} vs HGPS: p = {pval:.3g}",
            ha="center", va="top", fontsize=10,
            transform=ax.transAxes
        )
        y -= dy

    return y


# -----------------------------
# Mann–Whitney vs Controls for drugs
# -----------------------------
def add_mannwhitney_vs_controls(ax, pooled_df, value_col,
                               y_text_start, dy=0.12):

    baseline = pooled_df[pooled_df["Condition"] == "Controls"][value_col]
    drugs = ["Lonafarnib", "UCM-13207", "C75", "2 uM C75"]

    y = y_text_start
    for drug in drugs:
        drug_vals = pooled_df[pooled_df["Condition"] == drug][value_col]
        if len(drug_vals) == 0:
            continue

        stat, pval = mannwhitneyu(baseline, drug_vals, alternative="two-sided")

        ax.text(
            0.5, y,
            f"{drug} vs Controls: p = {pval:.3g}",
            ha="center", va="top", fontsize=10,
            transform=ax.transAxes
        )
        y -= dy

    return y

# -----------------------------
# Counts + medians text for all conditions
# -----------------------------
def add_counts_and_medians(ax, pooled_df, y_text_start=-0.65, dy=0.12):
    """
    Writes number of cells and medians for each condition under the axis.
    """
    conditions = ["Controls", "HGPS", "Lonafarnib", "UCM-13207", "C75", "2 uM C75"]

    y = y_text_start
    for cond in conditions:
        subset = pooled_df[pooled_df["Condition"] == cond]
        if len(subset) == 0:
            continue

        n = len(subset)
        med_D = subset["D_eff"].median()
        med_alpha = subset["alpha"].median()

        ax.text(
            0.5, y,
            f"{cond}: {n} cells, median D = {med_D:.3g}, median α = {med_alpha:.3g}",
            ha="center", va="top", fontsize=10,
            transform=ax.transAxes
        )
        y -= dy


# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects\no backlund\60 frames"

fit_csvs = {
    "Controls": [
        rf"{base}\Rep1_Controls_ALL_fit_params.csv",
        rf"{base}\Rep2_Controls_ALL_fit_params.csv",
        rf"{base}\Rep3_Controls_ALL_fit_params.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep2_HGPS_ALL_fit_params.csv",
        rf"{base}\Rep3_HGPS_ALL_fit_params.csv",
    ],
    "Lonafarnib": [
        rf"{base}\Rep1_Lonafarnib_ALL_fit_params.csv",
        rf"{base}\Rep2_Lonafarnib_ALL_fit_params.csv",
        rf"{base}\Rep3_Lonafarnib_ALL_fit_params.csv",
    ],
    "UCM-13207": [
        rf"{base}\Rep1_UCM_ALL_fit_params.csv",
        rf"{base}\Rep2_UCM_ALL_fit_params.csv",
        rf"{base}\Rep3_UCM_ALL_fit_params.csv",
    ],
    "C75": [
        rf"{base}\Rep1_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_C75_ALL_fit_params.csv",
    ],
    "2 uM C75": [
        rf"{base}\Rep1_2uM_C75_ALL_fit_params.csv",
        rf"{base}\Rep2_2uM_C75_ALL_fit_params.csv",
        rf"{base}\Rep3_2uM_C75_ALL_fit_params.csv",
    ]

}

# -----------------------------
# Pool fit CSVs into one DataFrame (track level)
# -----------------------------
all_tracks = []

for condition, files in fit_csvs.items():
    for i, fpath in enumerate(files, 1):
        df = pd.read_csv(fpath)
        df['Condition'] = condition
        df['Replicate'] = f"Rep{i}"
        all_tracks.append(df)

all_tracks = pd.concat(all_tracks, ignore_index=True)

# Quality filters
all_tracks = all_tracks[
    (all_tracks['D_eff'] > 1e-5) &
    (all_tracks['alpha'] > 0.01) &
    (all_tracks['sigma'] > 0.001)
]

# -----------------------------
# Median per cell
# -----------------------------
pooled_all = (
    all_tracks
    .groupby(['Condition', 'Replicate', 'cell_id'], as_index=False)
    .agg({
        'D_eff': 'median',
        'alpha': 'median',
        'sigma': 'median'
    })
)

# -----------------------------
# Function: plot violins + stats below x-axis
# -----------------------------
def plot_pooled_violins_with_stats(pooled_df, output_path):
    fig, axes = plt.subplots(1, 2, figsize=(12,6))

    pooled_df = pooled_df.copy()
    pooled_df['log_D_eff'] = np.log10(pooled_df['D_eff'])

    # -----------------------------
    # Mixed-effects models
    # -----------------------------
    stats_dict = {}
    for col in ['D_eff', 'alpha']:
        model = smf.mixedlm(f"{col} ~ Condition", pooled_df, groups=pooled_df["Replicate"]).fit()
        coef = model.params['Condition[T.HGPS]']
        ci_lower, ci_upper = model.conf_int().loc['Condition[T.HGPS]']
        stats_dict[col] = {'coef': coef, 'ci': (ci_lower, ci_upper)}

    # -----------------------------
    # Mann–Whitney on per-cell medians
    # -----------------------------
    mannwhitney_dict = {}
    for col in ['D_eff', 'alpha']:
        cond1 = pooled_df[pooled_df['Condition'] == 'Controls'][col]
        cond2 = pooled_df[pooled_df['Condition'] == 'HGPS'][col]
        stat, pval = mannwhitneyu(cond1, cond2, alternative='two-sided')
        mannwhitney_dict[col] = {'stat': stat, 'pval': pval}

    # -----------------------------
    # Count cells + medians of medians
    # -----------------------------
    counts = {}
    for cond in ['Controls', 'HGPS']:
        subset = pooled_df[pooled_df['Condition'] == cond]
        counts[cond] = {
            'cells': len(subset),
            'median_D': subset['D_eff'].median(),
            'median_alpha': subset['alpha'].median()
        }

    # -----------------------------
    # D_eff violin
    # -----------------------------
    sns.violinplot(y='log_D_eff', x='Condition', data=pooled_df, ax=axes[0],
                   inner='box', palette=["lightsteelblue", "slateblue"], edgecolor = "black", linewidth=0.9)
    sns.stripplot(y='log_D_eff', x='Condition', data=pooled_df, ax=axes[0],
                  color='k', size=3, alpha=0.25)

    axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)", fontsize=14)
    axes[0].set_title("Diffusion Coefficient", fontsize=16, fontweight='bold')
    axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter("$10^{{{x:.0f}}}$"))

    coef = stats_dict['D_eff']['coef']
    ci_lower, ci_upper = stats_dict['D_eff']['ci']
    pval = mannwhitney_dict['D_eff']['pval']
    axes[0].text(0.5, -0.3,
                 f"Mixed-effect coef: {coef:.3g}, 95% CI: [{ci_lower:.3g}, {ci_upper:.3g}]\n"
                 f"Mann–Whitney p = {pval:.3g}\n"
                 f"Controls: {counts['Controls']['cells']} cells, median D = {counts['Controls']['median_D']:.3g}\n"
                 f"HGPS: {counts['HGPS']['cells']} cells, median D = {counts['HGPS']['median_D']:.3g}",
                 ha='center', va='top', fontsize=10, transform=axes[0].transAxes)

    # -----------------------------
    # alpha violin
    # -----------------------------
    sns.violinplot(y='alpha', x='Condition', data=pooled_df, ax=axes[1],
                   inner='box', palette=["lightsteelblue", "slateblue"], edgecolor = "black", linewidth=0.9)
    sns.stripplot(y='alpha', x='Condition', data=pooled_df, ax=axes[1],
                  color='k', size=3, alpha=0.25)


    axes[1].set_ylabel(r"$\alpha$", fontsize=14)
    axes[1].set_title("Anomalous Exponent", fontsize=16, fontweight='bold')

    coef = stats_dict['alpha']['coef']
    ci_lower, ci_upper = stats_dict['alpha']['ci']
    pval = mannwhitney_dict['alpha']['pval']
    axes[1].text(0.5, -0.3,
                 f"Mixed-effect coef: {coef:.3g}, 95% CI: [{ci_lower:.3g}, {ci_upper:.3g}]\n"
                 f"Mann–Whitney p = {pval:.3g}\n"
                 f"Controls: {counts['Controls']['cells']} cells, median α = {counts['Controls']['median_alpha']:.3g}\n"
                 f"HGPS: {counts['HGPS']['cells']} cells, median α = {counts['HGPS']['median_alpha']:.3g}",
                 ha='center', va='top', fontsize=10, transform=axes[1].transAxes)

    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()
    plt.close(fig)

# -----------------------------
# Run Controls vs HGPS
# -----------------------------
output_file = rf"{base}\CellMedian_Controls_vs_HGPS_violins.svg"
plot_pooled_violins_with_stats(
    pooled_all[pooled_all["Condition"].isin(["Controls", "HGPS"])],
    output_file
)

# -----------------------------
# Run third figure: all conditions together + stats vs HGPS + counts
# -----------------------------
all_df = pooled_all[
    pooled_all["Condition"].isin([
        "Controls", "HGPS", "Lonafarnib", "UCM-13207", "C75", "2 uM C75"
    ])
].copy()

all_df["log_D_eff"] = np.log10(all_df["D_eff"])

all_palette = [
    "lightsteelblue",    # Controls
    "slateblue",         # HGPS
    "teal",              # Lonafarnib
    "mediumaquamarine",  # UCM-13207
    "mediumseagreen",    # C75
    "cadetblue"         # 2 uM C75
]

condition_order = [
    "Controls", "HGPS", "Lonafarnib", "UCM-13207", "C75", "2 uM C75"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 9))

# -----------------------------
# D_eff panel
# -----------------------------
sns.violinplot(
    y="log_D_eff", x="Condition", data=all_df, ax=axes[0],
    inner="box", palette=all_palette, order=condition_order
)
sns.stripplot(
    y="log_D_eff", x="Condition", data=all_df, ax=axes[0],
    color="k", size=3, alpha=0.25, order=condition_order
)

axes[0].margins(y=0.02)
axes[0].set_ylabel(r"$D_\mathrm{eff}$ ($\mu m^2/s^\alpha$)")
axes[0].set_title("Diffusion Coefficient (All Conditions)",
                  fontsize=16, fontweight="bold")
axes[0].yaxis.set_major_formatter(
    mticker.StrMethodFormatter("$10^{{{x:.0f}}}$")
)


# Mann–Whitney vs HGPS (D_eff)
y_after_mw_D = add_mannwhitney_vs_hgps(
    axes[0], all_df, "D_eff",
    y_text_start=-0.30, dy=0.12
)

# Mann–Whitney vs Controls (D_eff)
y_after_mw_D_ctrl = add_mannwhitney_vs_controls(
    axes[0], all_df, "D_eff",
    y_text_start=y_after_mw_D - 0.05, dy=0.12
)

# Counts below all MW blocks
add_counts_and_medians(
    axes[0], all_df,
    y_text_start=y_after_mw_D_ctrl - 0.05, dy=0.12
)


# -----------------------------
# Alpha panel
# -----------------------------
sns.violinplot(
    y="alpha", x="Condition", data=all_df, ax=axes[1],
    inner="box", palette=all_palette, order=condition_order
)
sns.stripplot(
    y="alpha", x="Condition", data=all_df, ax=axes[1],
    color="k", size=3, alpha=0.25, order=condition_order
)

axes[1].margins(y=0.02)
axes[1].set_ylabel(r"$\alpha$")
axes[1].set_title("Anomalous Exponent (All Conditions)",
                  fontsize=16, fontweight="bold")

# Mann–Whitney vs HGPS (alpha)
y_after_mw_alpha = add_mannwhitney_vs_hgps(
    axes[1], all_df, "alpha",
    y_text_start=-0.30, dy=0.12
)

# Mann–Whitney vs Controls (alpha)
y_after_mw_alpha_ctrl = add_mannwhitney_vs_controls(
    axes[1], all_df, "alpha",
    y_text_start=y_after_mw_alpha - 0.05, dy=0.12
)

# Counts below all MW blocks
add_counts_and_medians(
    axes[1], all_df,
    y_text_start=y_after_mw_alpha_ctrl - 0.05, dy=0.12
)


plt.tight_layout()
plt.subplots_adjust(bottom=0.55)
fig.savefig(
    rf"{base}\Pooled_All_Conditions_violins_with_stats.svg",
    dpi=300, bbox_inches="tight"
)
plt.show()
plt.close(fig)


In [ ]:
# -----------------------------
# Pool scan area CSVs with tracks
# -----------------------------
def pool_scan_areas_with_tracks(scan_csvs_dict):
    pooled_list = []
    for condition, files in scan_csvs_dict.items():
        for i, fpath in enumerate(files, 1):
            df = pd.read_csv(fpath)
            df = df.rename(columns={'scan_area_um2': 'scan_area'})
            df['Condition'] = condition
            df['Replicate'] = f"Rep{i}"
            pooled_list.append(df)
    pooled_df = pd.concat(pooled_list, ignore_index=True)

    # Collapse multiple frames per particle to one row per particle (mean scan area)
    pooled_df_unique = (
        pooled_df.groupby(['cell_id', 'particle', 'Condition', 'Replicate'], as_index=False)
        ['scan_area']
        .mean()
        .reset_index(drop=True)
    )

    # Count frames per particle as tracks
    tracks_per_particle = pooled_df.groupby(['cell_id', 'particle', 'Condition', 'Replicate']).size().reset_index(name='tracks')
    pooled_df_unique = pooled_df_unique.merge(tracks_per_particle, on=['cell_id', 'particle', 'Condition', 'Replicate'])
    
    return pooled_df_unique


# -----------------------------
# Plot scan areas with gradient color
# -----------------------------
def plot_scan_areas_gradient(pooled_df, output_path):
    fig, ax = plt.subplots(1, 1, figsize=(6,6))

    # Mixed-effects model
    model = smf.mixedlm("scan_area ~ Condition", pooled_df, groups=pooled_df["Replicate"]).fit()
    coef = model.params['Condition[T.HGPS]']
    ci_lower, ci_upper = model.conf_int().loc['Condition[T.HGPS]']

    # Mann–Whitney U
    cond1 = pooled_df[pooled_df['Condition'] == 'Controls']['scan_area']
    cond2 = pooled_df[pooled_df['Condition'] == 'HGPS']['scan_area']
    stat, pval = mannwhitneyu(cond1, cond2, alternative='two-sided')

    # Counts
    counts = {}
    for cond in ['Controls', 'HGPS']:
        subset = pooled_df[pooled_df['Condition'] == cond]
        n_particles = subset['particle'].nunique()
        n_cells = subset['cell_id'].nunique()
        n_tracks = subset['tracks'].sum()
        mean_area = subset['scan_area'].mean()
        counts[cond] = {'particles': n_particles, 'cells': n_cells, 'tracks': n_tracks, 'mean_area': mean_area}

    # Violin outline
    sns.violinplot(
        y='scan_area', x='Condition', data=pooled_df, ax=ax,
        inner=None, palette=["#DDDDDD", "#DDDDDD"], linewidth=1
    )

    # Points colored by scan area
    norm = mpl.colors.Normalize(vmin=pooled_df['scan_area'].min(), vmax=pooled_df['scan_area'].max())
    cmap = mpl.cm.viridis

    for i, cond in enumerate(['Controls', 'HGPS']):
        subset = pooled_df[pooled_df['Condition'] == cond]
        colors = cmap(norm(subset['scan_area']))
        ax.scatter(
            x=np.random.normal(i, 0.08, size=len(subset)),  # add jitter horizontally
            y=subset['scan_area'],
            c=colors,
            s=15,
            edgecolor='k',
            alpha=0.8
        )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label(r'Scan area ($\mu m^2$)', fontsize=12)

    ax.set_ylabel(r"Scan Area ($\mu m^2$)", fontsize=14)
    ax.set_title("Telomere Scan Area", fontsize=16, fontweight='bold')
    ax.grid(False)

    ax.text(0.5, -0.3,
            f"Mixed-effect coef: {coef:.3g}, 95% CI: [{ci_lower:.3g}, {ci_upper:.3g}]\n"
            f"Mann–Whitney p = {pval:.3g}\n"
            f"Controls: {counts['Controls']['cells']} cells, {counts['Controls']['particles']} particles, {counts['Controls']['tracks']} tracks, mean area = {counts['Controls']['mean_area']:.3g}\n"
            f"HGPS: {counts['HGPS']['cells']} cells, {counts['HGPS']['particles']} particles, {counts['HGPS']['tracks']} tracks, mean area = {counts['HGPS']['mean_area']:.3g}",
            ha='center', va='top', fontsize=10, transform=ax.transAxes)

    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()
    plt.close(fig)


# -----------------------------
# Run scan area plot
# -----------------------------
pooled_scan_areas_unique = pool_scan_areas_with_tracks(scan_csvs)
output_file = rf"{base}\Pooled_Controls_vs_HGPS_scan_areas_gradient.svg"
plot_scan_areas_gradient(pooled_scan_areas_unique, output_file)


### Bar Chart of Scan Area by Group

In [ ]:
def plot_scan_area_bar_chart():
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import sem

    groups = [
        "HGPS",
        "Control",
        "HGPS + Lonafarnib",
        "HGPS + UCM-13207",
        "HGPS + C75"
    ]

    data = {
        "HGPS": [0.196, 0.232, 0.331],
        "Control": [0.123, 0.092, 0.108],
        "HGPS + Lonafarnib": [0.142, 0.206, 0.160],
        "HGPS + UCM-13207": [0.217, 0.189, 0.216],
        "HGPS + C75": [0.086, 0.084, 0.132],
    }

    means = [np.mean(data[g]) for g in groups]
    errors = [sem(data[g]) for g in groups]

    colors = [
        "slateblue",  # HGPS
        "lightsteelblue",  # Control
        "teal",  # Lonafarnib
        "mediumaquamarine",  # UCM-13207
        "mediumseagreen"   # C75
        "darkslategray"   # 2 uM C75
    ]

    fig, ax = plt.subplots(figsize=(7, 5))
    x = np.arange(len(groups))

    ax.bar(
        x,
        means,
        yerr=errors,
        capsize=6,
        linewidth=1.5,
        color=colors
    )

    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=30, ha='right', fontsize=12)
    ax.set_ylabel("Scan area [µm²]", fontsize=14)
    ax.set_title("Bar Chart of Scan Area by Group", fontsize=16, fontweight='bold')

    ax.tick_params(axis='y', labelsize=12)
    ax.grid(False)

    plt.tight_layout()
    fig.savefig("D:\\Research Projects\\Progeria\\Chromatin Tracking\\Data\\20251231 Plotting Bar Graph\\scan_area_bar_chart.svg", dpi=300)
    plt.show()

    #Run
plot_scan_area_bar_chart()


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# -----------------------------
# File paths
# -----------------------------
base = r"D:\Research Projects\Progeria\Chromatin Tracking\Data\20260106 Checking for batch effects\no backlund\60 frames"

scan_area_csvs = {
    "Controls": [
        rf"{base}\Rep1_Controls_scan_areas.csv",
        rf"{base}\Rep2_Controls_scan_areas.csv",
        rf"{base}\Rep3_Controls_scan_areas.csv",
    ],
    "HGPS": [
        rf"{base}\Rep1_HGPS_scan_areas.csv",
        rf"{base}\Rep2_HGPS_scan_areas.csv",
        rf"{base}\Rep3_HGPS_scan_areas.csv",
    ],
    "2 uM Lonafarnib": [
        rf"{base}\Rep1_Lonafarnib_scan_areas.csv",
        rf"{base}\Rep2_Lonafarnib_scan_areas.csv",
        rf"{base}\Rep3_Lonafarnib_scan_areas.csv",
    ],
    "2 uM UCM-13207": [
        rf"{base}\Rep1_UCM_scan_areas.csv",
        rf"{base}\Rep2_UCM_scan_areas.csv",
        rf"{base}\Rep3_UCM_scan_areas.csv",
    ],
    "5 uM C75": [
        rf"{base}\Rep1_C75_scan_areas.csv",
        rf"{base}\Rep2_C75_scan_areas.csv",
        rf"{base}\Rep3_C75_scan_areas.csv",
    ],
    "2 uM C75": [
        rf"{base}\Rep1_2uM_C75_scan_areas.csv",
        rf"{base}\Rep2_2uM_C75_scan_areas.csv",
        rf"{base}\Rep3_2uM_C75_scan_areas.csv",
    ]
}

groups = [
    "HGPS",
    "Controls",
    "2 uM Lonafarnib",
    "2 uM UCM-13207",
    "5 uM C75",
    "2 uM C75"
]

colors = [
    "slateblue",
    "lightsteelblue",
    "teal",
    "mediumaquamarine",
    "mediumseagreen",
    "darkslategray"
]

# ==========================================================
# Helper functions
# ==========================================================
def get_per_cell_medians(fpaths):
    out = []
    for f in fpaths:
        df = pd.read_csv(f)
        for cid in df["cell_id"].unique():
            out.append(df[df["cell_id"] == cid]["scan_area_um2"].median())
    return np.array(out)

def get_per_cell_means(fpaths):
    out = []
    for f in fpaths:
        df = pd.read_csv(f)
        for cid in df["cell_id"].unique():
            out.append(df[df["cell_id"] == cid]["scan_area_um2"].mean())
    return np.array(out)

# ==========================================================
# Prepare per-cell data
# ==========================================================
data_per_cell_medians = {g: get_per_cell_medians(scan_area_csvs[g]) for g in groups}
data_per_cell_means   = {g: get_per_cell_means(scan_area_csvs[g]) for g in groups}

# ==========================================================
# Plot 1: mean ± SEM of per-cell MEDIANS
# ==========================================================
means_med = [np.mean(data_per_cell_medians[g]) for g in groups]
sems_med = [
    np.std(data_per_cell_medians[g], ddof=1) / np.sqrt(len(data_per_cell_medians[g]))
    for g in groups
]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(groups))

ax.bar(x, means_med, yerr=sems_med, capsize=6, color=colors, edgecolor="k")

bar_tops = [m + s for m, s in zip(means_med, sems_med)]
ax.set_ylim(0, max(bar_tops) * 1.35)

y_min = min([data_per_cell_medians[g].min() for g in groups])
y_range = max(bar_tops) - y_min
y_text = y_min - 0.25 * y_range
dy = 0.08 * y_range

# --- Mean ± SEM per group ---
for i, g in enumerate(groups):
    txt = f"{g}: mean={means_med[i]:.3g} ± {sems_med[i]:.3g}"
    ax.text(x[i], y_text, txt, ha="center", va="top", fontsize=10)
y_text -= 1.5 * dy

# --- vs HGPS ---
for i, g in enumerate(groups):
    if g != "HGPS":
        _, p = mannwhitneyu(
            data_per_cell_medians["HGPS"],
            data_per_cell_medians[g],
            alternative="two-sided"
        )
        txt = f"{g} vs HGPS: p={p:.3g}"
    else:
        txt = "HGPS (reference)"
    ax.text(x[i], y_text, txt, ha="center", va="top", fontsize=10)
    y_text -= dy

# --- vs Controls ---
y_text -= 1.5 * dy
for i, g in enumerate(groups):
    if g != "Controls":
        _, p = mannwhitneyu(
            data_per_cell_medians["Controls"],
            data_per_cell_medians[g],
            alternative="two-sided"
        )
        txt = f"{g} vs Controls: p={p:.3g}"
    else:
        txt = "Controls (reference)"
    ax.text(x[i], y_text, txt, ha="center", va="top", fontsize=10)
    y_text -= dy

ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=30, ha="right", fontsize=12)
ax.set_ylabel("Scan area [µm²]", fontsize=14)
ax.set_title(
    "Telomere Scan Areas per Condition\n(mean of per-cell MEDIANS ± SEM)",
    fontsize=16, fontweight="bold"
)

plt.subplots_adjust(bottom=0.5)
outpath = os.path.join(base, "scan_area_mean_of_per_cell_medians_bar_chart_only_sem.svg")
plt.savefig(outpath, dpi=300)
plt.show()

# ==========================================================
# Plot 2: mean ± SEM of per-cell MEANS
# ==========================================================
means_mean = [np.mean(data_per_cell_means[g]) for g in groups]
sems_mean = [
    np.std(data_per_cell_means[g], ddof=1) / np.sqrt(len(data_per_cell_means[g]))
    for g in groups
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x, means_mean, yerr=sems_mean, capsize=6, color=colors, edgecolor="k")

bar_tops = [m + s for m, s in zip(means_mean, sems_mean)]
ax.set_ylim(0, max(bar_tops) * 1.35)

y_min = min([data_per_cell_means[g].min() for g in groups])
y_range = max(bar_tops) - y_min
y_text = y_min - 0.25 * y_range
dy = 0.08 * y_range

# --- Mean ± SEM per group ---
for i, g in enumerate(groups):
    txt = f"{g}: mean={means_mean[i]:.3g} ± {sems_mean[i]:.3g}"
    ax.text(x[i], y_text, txt, ha="center", va="top", fontsize=10)
y_text -= 1.5 * dy

# --- vs HGPS ---
for i, g in enumerate(groups):
    if g != "HGPS":
        _, p = mannwhitneyu(
            data_per_cell_means["HGPS"],
            data_per_cell_means[g],
            alternative="two-sided"
        )
        txt = f"{g} vs HGPS: p={p:.3g}"
    else:
        txt = "HGPS (reference)"
    ax.text(x[i], y_text, txt, ha="center", va="top", fontsize=10)
    y_text -= dy

# --- vs Controls ---
y_text -= 1.5 * dy
for i, g in enumerate(groups):
    if g != "Controls":
        _, p = mannwhitneyu(
            data_per_cell_means["Controls"],
            data_per_cell_means[g],
            alternative="two-sided"
        )
        txt = f"{g} vs Controls: p={p:.3g}"
    else:
        txt = "Controls (reference)"
    ax.text(x[i], y_text, txt, ha="center", va="top", fontsize=10)
    y_text -= dy

ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=30, ha="right", fontsize=12)
ax.set_ylabel("Scan area [µm²]", fontsize=14)
ax.set_title(
    "Telomere Scan Areas per Condition\n(mean of per-cell MEANS ± SEM)",
    fontsize=16, fontweight="bold"
)

plt.subplots_adjust(bottom=0.5)
outpath = os.path.join(base, "scan_area_mean_of_per_cell_means_bar_chart_only_sem.svg")
plt.savefig(outpath, dpi=300)
plt.show()


### Recolor color bar for direct comparison of scan areas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from scipy.spatial import ConvexHull
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os

# ==============================
# User parameters
# ==============================
control_csv = "D:\\Research Projects\\Progeria\\Chromatin Tracking\\Data\\20251231 Replotting scan areas from replicate 2 on same color scale\\max distance 0.131x6\\60 frames\\Rep2_Controls_scan_areas.csv"
hgps_csv = "D:\\Research Projects\\Progeria\\Chromatin Tracking\\Data\\20251231 Replotting scan areas from replicate 2 on same color scale\\max distance 0.131x6\\60 frames\\Rep2_HGPS_scan_areas.csv"

# ==============================
# Extract folder for saving figures
# ==============================
outfolder = os.path.dirname(control_csv)
outprefix = os.path.join(outfolder, "scan_area_comparison")

# ==============================
# Load CSVs
# ==============================
control_df = pd.read_csv(control_csv)
hgps_df = pd.read_csv(hgps_csv)

# ==============================
# Gather all scan areas
# ==============================
def get_all_areas(df):
    areas = []
    for cell_id in df['cell_id'].unique():
        cell_tracks = df[df['cell_id'] == cell_id]
        for pid in cell_tracks['particle'].unique():
            areas.append(cell_tracks[cell_tracks['particle']==pid]['scan_area_um2'].values[0])
    return np.array(areas)
    
def count_tracks(df):
    return df[['cell_id', 'particle']].drop_duplicates().shape[0]

control_areas = get_all_areas(control_df)
hgps_areas = get_all_areas(hgps_df)

all_areas = np.concatenate([control_areas, hgps_areas])
vmin = np.percentile(all_areas, 5)
vmax = np.percentile(all_areas, 95)
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = cm.viridis

print(f"Shared colorbar range (5th–95th percentile): {vmin:.3f} – {vmax:.3f} µm²")
print(f"Control mean area: {control_areas.mean():.3f}, HGPS mean area: {hgps_areas.mean():.3f}")

# ==============================
# Plot scan areas function
# ==============================
def plot_scan_areas(df, title="Scan Areas", show_hull=True, save_path=None, padding_fraction=0.05):
    fig, ax = plt.subplots(figsize=(8,8))
    
    per_cell_areas = []
    all_points = []  # collect all points
    for cell_id in df['cell_id'].unique():
        cell_tracks = df[df['cell_id']==cell_id]
        for pid in cell_tracks['particle'].unique():
            track = cell_tracks[cell_tracks['particle'] == pid].sort_values('frame')
            points = track[['x','y']].values
            all_points.append(points)
            scan_area = track['scan_area_um2'].values[0]
            per_cell_areas.append(scan_area)
            color = cmap(norm(scan_area))
            
            # Trajectory
            ax.plot(track['x'], track['y'], color=color, linewidth=1.5, alpha=0.9)
            
            # Hull
            if show_hull and len(points) >= 3:
                hull = ConvexHull(points)
                hull_vertices = points[hull.vertices]
                ax.fill(hull_vertices[:,0], hull_vertices[:,1], color=color, alpha=0.15)
    
    # --- Make plot square ---
    all_points = np.vstack(all_points)
    xmin, xmax = all_points[:,0].min(), all_points[:,0].max()
    ymin, ymax = all_points[:,1].min(), all_points[:,1].max()
    x_center, y_center = (xmin + xmax)/2, (ymin + ymax)/2
    half_range = max(xmax - xmin, ymax - ymin)/2 * (1 + padding_fraction)
    ax.set_xlim(x_center - half_range, x_center + half_range)
    ax.set_ylim(y_center - half_range, y_center + half_range)
    
    # Plot mean 
    mean_area = np.mean(per_cell_areas)
    ax.text(0.98, 0.98, f"Mean scan area:\n{mean_area:.3f} µm²",
            transform=ax.transAxes, ha='right', va='top', fontsize=12)
    
    ax.set_aspect('equal')
    ax.set_xlabel("x [µm]", fontsize=14)
    ax.set_ylabel("y [µm]", fontsize=14)
    ax.tick_params(axis='both', labelsize=12)
    ax.set_title(title, fontsize=16, fontweight='bold')
    
    # Colorbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label("Scan area [µm²]", fontsize=14)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f"Saved figure: {save_path}")
    plt.show()


# ==============================
# Plot Control
# ==============================
plot_scan_areas(control_df, title="Control Telomere Scan Areas",
                show_hull=True, save_path=os.path.join(outfolder, "scan_area_comparison_control_hull.svg"))
plot_scan_areas(control_df, title="Control Telomere Scan Areas (no hull)",
                show_hull=False, save_path=os.path.join(outfolder, "scan_area_comparison_control_no_hull.svg"))

# ==============================
# Plot HGPS
# ==============================
plot_scan_areas(hgps_df, title="HGPS Cells Scan Areas",
                show_hull=True, save_path=os.path.join(outfolder, "scan_area_comparison_hgps_hull.svg"))
plot_scan_areas(hgps_df, title="HGPS Cells Scan Areas (no hull)",
                show_hull=False, save_path=os.path.join(outfolder, "scan_area_comparison_hgps_no_hull.svg"))

n_control_tracks = count_tracks(control_df)
n_hgps_tracks = count_tracks(hgps_df)

print(f"Control image shows {n_control_tracks} tracks")
print(f"HGPS image shows {n_hgps_tracks} tracks")


### Cell Counting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Output directory
save_dir = r"D:\Research Projects\Progeria\CellCounting"
png_path = os.path.join(save_dir, "cell_growth_curves_normalized.png")
svg_path = os.path.join(save_dir, "cell_growth_curves_normalized.svg")

# Days
days = np.arange(1, 8)

# Data (each entry: days × 3 FOVs)

Control = [
    [68, 74, 92],
    [120, 124, 121],
    [243, 254, 266],
    [374, 433, 421],
]

C75_5 = [
    [300, 352, 313],
    [365, 354, 419],
    [374, 431, 499],
    [532, 444, 506],
    [502, 540, 591],
    [603, 577, 594],
    [692, 619, 639],
]

C75_2 = [
    [111, 130, 137],
    [139, 169, 136],
    [169, 160, 170],
    [185, 177, 176],
    [152, 195, 243],
    [240, 216, 201],
    [238, 247, 253],
]

HGPS = [
    [114, 108, 87],
    [113, 112, 118],
    [121, 143, 123],
    [110, 157, 152],
    [153, 175, 120],
    [163, 156, 168],
    [165, 173, 179],
]

UCM2 = [
    [49, 48, 43],
    [47, 56, 52],
    [55, 56, 61],
    [65, 57, 63],
    [65, 74, 67],
    [68, 78, 70],
    [81, 73, 74],
]

Lonafarnib = [
    [131, 128, 131],
    [133, 140, 134],
    [152, 115, 158],
    [155, 158, 140],
    [160, 159, 148],
    [176, 161, 165],
    [171, 170, 180],
]

conditions = [Control, C75_5, C75_2, HGPS, UCM2, Lonafarnib]
names = [
    "Control",
    "5 µM C75",
    "2 µM C75",
    "HGPS",
    "2 µM UCM-13207",
    "2 µM Lonafarnib",
]

colors = [
    "lightsteelblue",
    "mediumseagreen",
    "darkslategray",
    "slateblue",
    "mediumaquamarine",
    "teal",
]


# Normalize to Day 1 mean
norm_mean = mean_vals / mean_vals[:, [0]]
norm_sem  = sem_vals  / mean_vals[:, [0]]

# Plot
plt.figure(figsize=(6.5, 4.5))

for cond, name, color in zip(conditions, names, colors):
    data = np.array(cond)              # shape: (n_days, n_FOVs)
    n_days = data.shape[0]

    mean_vals = data.mean(axis=1)
    sem_vals  = data.std(axis=1, ddof=1) / np.sqrt(data.shape[1])

    # Normalize to Day 1
    norm_mean = mean_vals / mean_vals[0]
    norm_sem  = sem_vals  / mean_vals[0]

    plt.errorbar(
        days[:n_days],                 # <-- match days to data length
        norm_mean,
        yerr=norm_sem,
        marker="o",
        linewidth=1.8,
        markersize=5,
        capsize=3,
        color=color,
        label=name,
    )


plt.xlabel("Day")
plt.ylabel("Fold change relative to Day 1")
plt.title("Cell growth curves (daily mean ± SEM, normalized)")
plt.legend(frameon=False)

plt.tight_layout()

# Save figures
plt.savefig(png_path, dpi=600, bbox_inches="tight")
plt.savefig(svg_path, bbox_inches="tight")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Output directory
save_dir = r"D:\Research Projects\Progeria\CellCounting"
png_path = os.path.join(save_dir, "cell_growth_curves_normalized_nocontrols.png")
svg_path = os.path.join(save_dir, "cell_growth_curves_normalized_nocontrols.svg")

# Days
days = np.arange(1, 8)

# Data (each entry: days × 3 FOVs)

C75_5 = [
    [300, 352, 313],
    [365, 354, 419],
    [374, 431, 499],
    [532, 444, 506],
    [502, 540, 591],
    [603, 577, 594],
    [692, 619, 639],
]

C75_2 = [
    [111, 130, 137],
    [139, 169, 136],
    [169, 160, 170],
    [185, 177, 176],
    [152, 195, 243],
    [240, 216, 201],
    [238, 247, 253],
]

HGPS = [
    [114, 108, 87],
    [113, 112, 118],
    [121, 143, 123],
    [110, 157, 152],
    [153, 175, 120],
    [163, 156, 168],
    [165, 173, 179],
]

UCM2 = [
    [49, 48, 43],
    [47, 56, 52],
    [55, 56, 61],
    [65, 57, 63],
    [65, 74, 67],
    [68, 78, 70],
    [81, 73, 74],
]

Lonafarnib = [
    [131, 128, 131],
    [133, 140, 134],
    [152, 115, 158],
    [155, 158, 140],
    [160, 159, 148],
    [176, 161, 165],
    [171, 170, 180],
]

conditions = [C75_5, C75_2, HGPS, UCM2, Lonafarnib]
names = [
    "5 µM C75",
    "2 µM C75",
    "HGPS",
    "2 µM UCM-13207",
    "2 µM Lonafarnib",
]

colors = [
    "mediumseagreen",
    "darkslategray",
    "slateblue",
    "mediumaquamarine",
    "teal",
]

# Convert to numpy for calculation
mean_vals = np.zeros((len(conditions), len(days)))
sem_vals  = np.zeros_like(mean_vals)

for i, cond in enumerate(conditions):
    data = np.array(cond)
    mean_vals[i] = data.mean(axis=1)
    sem_vals[i]  = data.std(axis=1, ddof=1) / np.sqrt(data.shape[1])

# Normalize to Day 1 mean
norm_mean = mean_vals / mean_vals[:, [0]]
norm_sem  = sem_vals  / mean_vals[:, [0]]

# Plot
plt.figure(figsize=(6.5, 4.5))

for i in range(len(conditions)):
    plt.errorbar(
        days,
        norm_mean[i],
        yerr=norm_sem[i],
        marker="o",
        linewidth=1.8,
        markersize=5,
        capsize=3,
        color=colors[i],
        label=names[i],
    )

plt.xlabel("Day")
plt.ylabel("Fold change relative to Day 1")
plt.title("Cell growth curves (daily mean ± SEM, normalized)")
plt.legend(frameon=False)

plt.tight_layout()

# Save figures
plt.savefig(png_path, dpi=600, bbox_inches="tight")
plt.savefig(svg_path, bbox_inches="tight")

plt.show()
